# Atomiq 2026 — Real IBM Quantum Hardware Experiments (FINAL)

**Qiskit Fall Fest 2026 · Atomiq · Stage 3 hardware validation with a hardware-sized, clearly synthetic Stage 2 study**

This notebook runs the quantum parts of the Atomiq project on IBM Quantum hardware while protecting the free Open Plan allowance (about 10 QPU minutes per rolling 28 days). Everything is validated locally first. Hardware jobs are never submitted by "Run all".

| Experiment | What it does | Data |
|---|---|---|
| **A** (priority 1) | Stage 3: six-qubit independent-Bernoulli circuit, estimate $P(K\ge 3)$ | **Real** Stage 3 probabilities from the project (verified against the reference value) |
| **B** (priority 2) | Exact-cardinality XY-mixer QAOA, choose 3 of 6 | **SYNTHETIC** instance. Not the Atomiq challenge data. |
| **C** | Physical-qubit mapping comparison | Same circuits as A and B |
| **D** | Dynamical decoupling, twirling, readout mitigation | A and B |
| **E** | Run-to-run repeatability of Experiment A | A |

### The Atomiq problem in brief
* **Stage 1** gives each customer a subscription probability $p_i$ (frozen, not recomputed here).
* **Stage 2** (actual challenge) picks exactly 6 of the top-10 customers. This is solved **classically and exactly** (210 selections enumerated, matched by a SciPy MILP). The old ten-qubit XY-QAOA circuit (depth about 6,010, about 3,081 CX, about 981 routing SWAPs) is **not** sent to hardware.
* **Stage 3** treats the six chosen customers `C0028, C0031, C0079, C0081, C0327, C0338` as independent Bernoulli variables and asks for $P(K\ge 3)$, where $K=\sum_i X_i$. Validated exact reference: $0.2078515973394121$.

### Stage 3 quantum encoding
$$R_Y(\theta_i)\lvert 0\rangle=\sqrt{1-p_i}\,\lvert 0\rangle+\sqrt{p_i}\,\lvert 1\rangle,\qquad \theta_i = 2\arcsin\sqrt{p_i}.$$
The six qubits are never entangled, so the circuit has no two-qubit gates. Its hardware error is dominated by readout.

### How to run (read before running anything)
1. **Simulation only (default).** Add nothing, run all cells. Zero IBM jobs are submitted. With an API key the notebook connects read-only to use real backend calibration; without one it uses a local fake-backend snapshot.
2. **Stage 3 data.** The notebook searches for the project's Stage 3 probabilities file and refuses to continue unless the exact tail probability equals the validated reference. If it cannot find the file it tells you what is missing (upload it when prompted, or set `STAGE3_INPUT_FILE`).
3. **Hardware (explicit opt-in).** Add Colab Secret `IBM_QUANTUM_API_KEY` (and optionally `IBM_QUANTUM_INSTANCE`). Read your **remaining QPU seconds from the IBM Quantum dashboard** and set `REMAINING_QPU_SECONDS`. Then set `ENABLE_HARDWARE = True`, `HARDWARE_CONFIRMATION_TEXT`, and add the job label to `CONFIRMED_JOBS`. Each submission also asks for a typed `YES`. At most `MAX_NEW_JOBS_PER_RUN` (default 1) jobs are submitted per run.
4. After a disconnect, re-run all cells. Jobs recorded in the registry are resumed from their job IDs, never resubmitted.

*Nothing in this notebook prints, stores or saves your API key.*

> **Published source note:** this notebook is the executable source, with outputs cleared to avoid presenting stale Colab outputs as a fresh run. Verified execution evidence is under `results/ibm_hardware/`. The archived source export is preserved unchanged. Its resumed Stage 3 job record incorrectly inherited the later Kingston backend/layout; the canonical audit corrects that attribution to the original Marrakesh execution and documents the basis. The QAOA p=1/p=2 job is correctly attributed to Kingston. No API key is embedded.


## 1. Environment setup
Versions of the Qiskit stack are pinned to a tested combination. If the pip cell changes anything, restart the runtime once before continuing.

In [ ]:
# Tested combination (Python 3.12/3.13). Qiskit versions are pinned, and python-dateutil
# is constrained below 3 to remain compatible with Colab packages (BigQuery, ibis, holidays, etc.).
%pip install -q "python-dateutil>=2.9.0.post0,<3" "qiskit==2.2.3" "qiskit-aer==0.17.2" "qiskit-ibm-runtime==0.43.1"

## 2. Imports and experiment configuration

In [ ]:
import hashlib
import json
import math
import os
import platform
import sys
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from scipy.linalg import expm

import qiskit
import qiskit_aer
import qiskit_ibm_runtime
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

print(f"Python {platform.python_version()} | qiskit {qiskit.__version__} | "
      f"qiskit-aer {qiskit_aer.__version__} | qiskit-ibm-runtime {qiskit_ibm_runtime.__version__}")

TESTED_VERSIONS = {"qiskit": "2.2.3", "qiskit_aer": "0.17.2", "qiskit_ibm_runtime": "0.43.1"}
for package_name, module in (("qiskit", qiskit), ("qiskit_aer", qiskit_aer), ("qiskit_ibm_runtime", qiskit_ibm_runtime)):
    if module.__version__ != TESTED_VERSIONS[package_name]:
        print(f"WARNING: {package_name} {module.__version__} differs from the tested {TESTED_VERSIONS[package_name]}. "
              "Restart the runtime after the pip cell, or expect API differences.")

In [ ]:
# ----------------------------------------------------------------------------
# Experiment configuration. Edit values here; nothing else needs changing.
# ----------------------------------------------------------------------------

# ===== HARDWARE SAFETY GATES (all must be satisfied before any job is submitted) =====
ENABLE_HARDWARE = False                       # disarmed by default; enable only for a deliberate new hardware run
REQUIRED_CONFIRMATION_TEXT = "SUBMIT TO IBM HARDWARE"
HARDWARE_CONFIRMATION_TEXT = ""                # set the exact required phrase only for a deliberate new hardware run
REMAINING_QPU_SECONDS = None                  # use authenticated service.usage() first; manual input is fallback only
CONFIRMED_JOBS = set()                         # explicitly list approved jobs only after reviewing the budget
MAX_NEW_JOBS_PER_RUN = 2                      # hard cap: at most Stage 3 plus one QAOA job per run
REQUIRE_INTERACTIVE_CONFIRMATION = True       # 6. typed YES before each submission
# Set True only if REMAINING_QPU_SECONDS was read from the dashboard AFTER all earlier jobs
# in job_registry.json had finished and been billed. Otherwise earlier jobs are subtracted again.
BALANCE_INCLUDES_PREVIOUS_JOBS = False
MANUAL_USAGE_OVERRIDES = {}                   # {ibm_job_id: qpu_seconds} copied from the dashboard
RESUME_JOB_IDS = {"job1_stage3_best_layout": "dbqfi99abdes73fo6vig"}  # collect the completed Stage 3 job; never resubmits it
# Provenance captured when the job was originally submitted. Never infer these fields from the
# currently selected backend: a resumed job may belong to a prior backend/layout.
RESUME_JOB_METADATA = {
    "job1_stage3_best_layout": {
        "backend": "ibm_marrakesh",
        "physical_qubits_by_label": {
            "A_best": [87, 6, 10, 139, 120, 107],
            "cal0_A_best": [87, 6, 10, 139, 120, 107],
            "cal1_A_best": [87, 6, 10, 139, 120, 107],
        },
    },
}

# ===== Budget protection =====
QPU_RESERVE_FRACTION = 0.30                   # never plan to spend the last 30% of the remaining balance
QPU_MIN_RESERVE_SECONDS = 120.0               # ... or the last 120 s, whichever is larger
BUDGET_UNCERTAINTY_FACTOR = 2.0               # planned cost is inflated by this factor
MAX_PLANNED_SECONDS_PER_JOB = 60.0            # no single job may be planned above this
FIXED_JOB_OVERHEAD_SECONDS = 3.0              # per-job overhead in the estimate
CIRCUIT_OVERHEAD_SECONDS = 0.5                # per-circuit overhead in the estimate
FALLBACK_CIRCUIT_DURATION_SECONDS = 200e-6    # used when a circuit duration cannot be estimated
MAX_QUEUE_WAIT_MINUTES = 30                   # stop waiting after this; the job keeps running on IBM
POLL_SECONDS = 10

# ===== Credentials (names of Colab Secrets; the key itself is never handled here) =====
API_KEY_SECRET_NAME = "IBM_QUANTUM_API_KEY"
INSTANCE_SECRET_NAME = "IBM_QUANTUM_INSTANCE"      # optional

# ===== Stage 3 input =====
# None = search the usual project locations (see section 6). Or point at one file.
STAGE3_INPUT_FILE = "results/stage3_final/actual_stage3_results.json"  # canonical validated project artifact
STAGE3_SEARCH_DIRS = [".", "/content", "/content/drive/MyDrive", "/mnt/user-data/uploads"]
MANUAL_PROBABILITIES = None          # explicit override; still subject to the reference check
MANUAL_CUSTOMER_IDS = None

# Validated mathematical reference (never modified by this notebook).
EXACT_REFERENCE_TAIL = 0.2078515973394121
REFERENCE_TOLERANCE = 1e-9
EXPECTED_CUSTOMER_IDS = ["C0028", "C0031", "C0079", "C0081", "C0327", "C0338"]
NUM_CUSTOMERS = 6
SUCCESS_THRESHOLD = 3                               # P(K >= 3)

RANDOM_SEED = 2026

# ===== Backend / shots =====
PREFERRED_BACKEND_NAME = None        # e.g. "ibm_fez"; None = least busy (or the backend of an existing job)
MIN_BACKEND_QUBITS = 20
SHOTS_STAGE3 = 8192                  # per Experiment-A circuit
SHOTS_CALIBRATION = 2048             # per readout-calibration circuit
SHOTS_QAOA = 4096                    # existing simulation baseline
SHOTS_QAOA_HARDWARE = 30000         # equal hardware shots for p=1 and p=2
SHOTS_QAOA_HARDWARE_CALIBRATION = 8000  # per calibration state, per depth
REPEATABILITY_RUNS = 1               # extra independent jobs for Experiment E

# ===== Experiment B (SYNTHETIC) =====
QAOA_LAYERS = 1
SYNTHETIC_PROBLEM_SEED = 1           # chosen so that greedy != exact optimum
SELECT_COUNT = 3                     # choose 3 of 6 (exact cardinality)

# ===== Output =====
USE_GOOGLE_DRIVE = False             # True: store results on Drive so they survive a runtime reset
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/atomiq/results/ibm_hardware"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
else:
    OUTPUT_DIR = Path("results/ibm_hardware")
RAW_DIR = OUTPUT_DIR / "raw"
PROCESSED_DIR = OUTPUT_DIR / "processed"
FIGURE_DIR = OUTPUT_DIR / "figures"
for folder in (RAW_DIR, PROCESSED_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(RANDOM_SEED)
START_TIME_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Colour-blind-safe palette used by every figure.
COLORS = {"exact": "#000000", "ideal": "#0072B2", "noisy": "#E69F00",
          "hardware": "#D55E00", "mitigated": "#009E73", "suppressed": "#CC79A7",
          "neutral": "#7F7F7F"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})


def to_jsonable(value):
    """Convert numpy / pathlib objects so that json.dump never fails."""
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (set, tuple)):
        return list(value)
    raise TypeError(f"Cannot serialise {type(value).__name__}")


def save_json(obj, path):
    """Write JSON atomically so a crash cannot leave a half-written registry."""
    path = Path(path)
    temp_path = path.with_name(path.name + ".tmp")
    with open(temp_path, "w") as handle:
        json.dump(obj, handle, indent=2, default=to_jsonable)
    os.replace(temp_path, path)


def save_figure(fig, name):
    """Save a figure as high-resolution PNG and PDF."""
    fig.savefig(FIGURE_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
    print(f"saved figure: {name}")

## 3. Secure IBM Quantum authentication
The API key is read from the Colab Secret **`IBM_QUANTUM_API_KEY`** (or from an environment variable of the same name outside Colab). It is passed straight to `QiskitRuntimeService`; it is never printed, never written to disk (`save_account` is deliberately not used) and never included in any saved artifact.

In [ ]:
def read_secret(name, *, required=True):
    """Return a Colab secret (or environment variable); None if unavailable."""
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get(name) or None
    try:
        return userdata.get(name) or None
    except Exception as error:      # SecretNotFoundError, NotebookAccessError, ...
        if required:
            print(f"Colab secret '{name}' not available ({type(error).__name__}).")
        return None


def connect_to_ibm_quantum():
    """Create a QiskitRuntimeService, or return None with clear guidance."""
    token = read_secret(API_KEY_SECRET_NAME)
    if token is None:
        print(f"No IBM Quantum API key found. Add a Colab Secret named '{API_KEY_SECRET_NAME}' "
              "and enable notebook access. Continuing in simulation-only mode.")
        return None
    connection_options = {"channel": "ibm_quantum_platform", "token": token}
    instance = read_secret(INSTANCE_SECRET_NAME, required=False)
    if instance:
        connection_options["instance"] = instance
    try:
        service = QiskitRuntimeService(**connection_options)
    except Exception as error:
        message = str(error).replace(token, "***")      # make sure the key cannot leak
        if instance:
            message = message.replace(instance, "***")
        print(f"IBM Quantum login failed ({type(error).__name__}): {message}")
        return None
    print("Connected to IBM Quantum.")
    return service


service = connect_to_ibm_quantum()

def hardware_connection_preflight(service):
    """Read-only check: authenticate, identify the active instance, and list real QPUs. Never submits work."""
    if service is None:
        print("HARDWARE PREFLIGHT: FAIL — IBM Runtime authentication unavailable; no QPU job was submitted.")
        return False, None, []
    try:
        active_instance = service.active_instance()
        accessible_qpus = service.backends(simulator=False, operational=True,
                                           min_num_qubits=MIN_BACKEND_QUBITS)
    except Exception as error:
        print(f"HARDWARE PREFLIGHT: FAIL — IBM account/backend check raised {type(error).__name__}; no QPU job was submitted.")
        return False, None, []
    if not accessible_qpus:
        print(f"HARDWARE PREFLIGHT: FAIL — authenticated, but no operational non-simulator QPU with at least {MIN_BACKEND_QUBITS} qubits is accessible; no job was submitted.")
        return False, active_instance, []
    names = sorted(backend.name for backend in accessible_qpus)
    print("HARDWARE PREFLIGHT: PASS — IBM Runtime authenticated; active instance identified; "
          f"{len(names)} operational non-simulator QPU(s) accessible: {names}. Read-only check only; no job submitted.")
    return True, active_instance, accessible_qpus


HARDWARE_PREFLIGHT_OK, ACTIVE_IBM_INSTANCE, PREFLIGHT_BACKENDS = hardware_connection_preflight(service)

## 4. Read-only IBM hardware preflight and backend selection

Before any data processing or experiment, the previous cell authenticates with IBM, identifies the active instance, and lists accessible operational non-simulator QPUs. This is a read-only API check: it does not submit a job or consume QPU time. Backend discovery reuses that list. A successful check confirms account access and backend visibility, not a completed hardware experiment.


In [ ]:
TWO_QUBIT_GATE_CANDIDATES = ("cz", "ecr", "cx")


def find_two_qubit_gate(target):
    for name in TWO_QUBIT_GATE_CANDIDATES:
        if name in target.operation_names:
            return name
    raise RuntimeError("Backend has no supported native two-qubit gate (cz/ecr/cx).")


def median_two_qubit_error(backend):
    target = backend.target
    gate = find_two_qubit_gate(target)
    errors = [props.error for props in target[gate].values()
              if props is not None and props.error is not None]
    return float(np.median(errors)) if errors else float("nan")


def discover_backends(service):
    rows = []
    for backend in PREFLIGHT_BACKENDS:
        try:
            rows.append({"backend": backend.name, "qubits": backend.num_qubits,
                         "pending_jobs": backend.status().pending_jobs,
                         "median_2q_error": median_two_qubit_error(backend),
                         "object": backend})
        except Exception as error:
            print(f"Skipping {backend.name}: {type(error).__name__}")
    if not rows:
        raise RuntimeError("No operational IBM backend with enough qubits was found.")
    return pd.DataFrame(rows)


def load_fake_backend():
    """Local backend snapshot used when there is no IBM login (clearly labelled fake)."""
    from qiskit_ibm_runtime import fake_provider
    for class_name in ("FakeFez", "FakeTorino", "FakeSherbrooke", "FakeBrisbane"):
        backend_class = getattr(fake_provider, class_name, None)
        if backend_class is not None:
            return backend_class()
    raise RuntimeError("No fake backend is available in this qiskit-ibm-runtime version.")


def peek_registry_backend():
    """Backend of an already-submitted job, so later jobs and resumed jobs stay on the same device."""
    registry_file = RAW_DIR / "job_registry.json"
    try:
        with open(registry_file) as handle:
            names = {r.get("backend") for r in json.load(handle).values() if r.get("backend")}
    except (OSError, json.JSONDecodeError):
        return None
    return next(iter(names)) if len(names) == 1 else None


PINNED_BACKEND_NAME = PREFERRED_BACKEND_NAME or peek_registry_backend()


backend_table = None
backend_is_fake = service is None
if service is not None:
    backend_table = discover_backends(service)
    display_table = backend_table.drop(columns="object").sort_values(["pending_jobs", "median_2q_error"])
    print(display_table.to_string(index=False))
    if PINNED_BACKEND_NAME:
        chosen = backend_table[backend_table.backend == PINNED_BACKEND_NAME]
        if chosen.empty:
            raise ValueError(f"Backend '{PINNED_BACKEND_NAME}' is not available to this account.")
    else:
        chosen = backend_table.sort_values(["pending_jobs", "median_2q_error"]).head(1)
    backend = chosen.iloc[0]["object"]
else:
    backend = load_fake_backend()

print(f"\nSelected backend: {backend.name} ({backend.num_qubits} qubits) "
      f"{'[FAKE LOCAL SNAPSHOT - no hardware jobs will be sent]' if backend_is_fake else '[real device]'}")

## 5. Backend calibration and hardware inspection
We read the calibration data stored in the backend's `Target`: per-qubit readout error, single-qubit gate error, $T_1$/$T_2$, and per-pair two-qubit gate error. We also check the queue and, where the account supports it, the remaining QPU time.

In [ ]:
def instruction_error(target, gate_name, qubits):
    """Error of one gate on given qubits; NaN if not defined."""
    if gate_name not in target.operation_names:
        return float("nan")
    properties = target[gate_name].get(tuple(qubits))
    if properties is None or properties.error is None:
        return float("nan")
    return float(properties.error)


def collect_calibration(backend):
    """Return (per-qubit DataFrame, per-pair DataFrame, two-qubit gate name)."""
    target = backend.target
    gate = find_two_qubit_gate(target)
    qubit_rows = []
    for qubit in range(target.num_qubits):
        qubit_properties = target.qubit_properties[qubit] if target.qubit_properties else None
        t1 = getattr(qubit_properties, "t1", None)
        t2 = getattr(qubit_properties, "t2", None)
        qubit_rows.append({
            "qubit": qubit,
            "readout_error": instruction_error(target, "measure", (qubit,)),
            "sx_error": instruction_error(target, "sx", (qubit,)),
            "t1_us": t1 * 1e6 if t1 is not None else float("nan"),
            "t2_us": t2 * 1e6 if t2 is not None else float("nan"),
        })
    edge_rows = [{"qubit_a": a, "qubit_b": b, "gate": gate, "error": instruction_error(target, gate, (a, b))}
                 for (a, b) in target[gate].keys()]
    return pd.DataFrame(qubit_rows), pd.DataFrame(edge_rows), gate


qubit_calibration, edge_calibration, TWO_QUBIT_GATE = collect_calibration(backend)
operational_qubits = qubit_calibration.dropna(subset=["readout_error"])

print(f"Native two-qubit gate: {TWO_QUBIT_GATE}")
print(f"Basis gates: {sorted(backend.target.operation_names)}")
print(f"Operational qubits with readout calibration: {len(operational_qubits)} / {len(qubit_calibration)}")
print(f"Median readout error      : {operational_qubits.readout_error.median():.4f}")
print(f"Median sx error           : {operational_qubits.sx_error.median():.2e}")
print(f"Median {TWO_QUBIT_GATE} error        : {edge_calibration.error.median():.4f}")
print(f"Median T1 / T2 (us)       : {operational_qubits.t1_us.median():.0f} / {operational_qubits.t2_us.median():.0f}")

if not backend_is_fake:
    print(f"Queue: {backend.status().pending_jobs} pending jobs, status message: {backend.status().status_msg}")


def read_account_usage(service):
    """Remaining QPU seconds reported by the account API, or None. Informational: the dashboard decides."""
    if service is None:
        return None
    try:
        usage = service.usage()
    except Exception as error:
        print(f"Account usage lookup unavailable ({type(error).__name__}). Use the IBM Quantum dashboard value.")
        return None
    if not isinstance(usage, dict):
        print("Account usage lookup returned an unexpected format; ignoring it.")
        return None
    shown_keys = ("usage_remaining_seconds", "usage_consumed_seconds", "usage_limit_seconds",
                  "usage_allocation_seconds", "usage_period")
    print("Account usage report:", {key: usage[key] for key in shown_keys if key in usage})
    remaining = usage.get("usage_remaining_seconds")
    return float(remaining) if is_number(remaining) else None


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


account_remaining_seconds = read_account_usage(service)

## 6. Loading the actual Atomiq Stage 3 probabilities
The six probabilities must be the **frozen Stage 1 probabilities of the six customers chosen in Stage 2** (`C0028, C0031, C0079, C0081, C0327, C0338`). They are loaded from the project files, never changed and never invented.

**Search order:** `MANUAL_PROBABILITIES` (if set), then the canonical `results/stage3_final/actual_stage3_results.json`, or the explicitly configured `STAGE3_INPUT_FILE`. If unavailable, the notebook searches `.json`, `.csv`, and `.tsv` files under `STAGE3_SEARCH_DIRS` (depth limit 3), excluding its own hardware-output directory. In Colab, if no valid source is found you are asked to upload one.

**Accepted layouts:** a JSON list of six numbers; a JSON object with probability arrays and customer-ID arrays; an object mapping customer ID to probability; or records/CSV/TSV with customer-ID and probability columns. The canonical result records use `customer_id` and `stage1_success_probability`. A six-row ID-bearing candidate must contain exactly the six expected IDs, with no duplicates or unexpected IDs. A larger customer table may include other customers, but IDs must be unique and each expected ID must occur exactly once. In every ID-bearing case, values are mapped by customer ID and returned in the exact validated order above, regardless of input row order.

A six-value source without customer IDs may be used for **simulation only** after probability-range and exact-reference checks. Its customer mapping is unverified; it is never marked verified and the hardware gate must block submission. Duplicate, missing, or incorrect IDs in an ID-bearing candidate are rejected.

**A candidate is accepted only if all checks pass:** exactly six selected values, each finite and strictly in (0, 1), and the exact tail probability equal to the validated reference within `REFERENCE_TOLERANCE`. The tolerance is not loosened. If the file stores rounded probabilities the check fails and full-precision values are required.

If no source passes, the notebook **stops here** and names what is missing. It does not substitute other probabilities.

In [ ]:
PROBABILITY_KEYS = ("stage3_probabilities", "probabilities", "probability", "prob", "proba", "p",
                    "predicted_probability", "subscription_probability", "stage1_probability", "stage1_success_probability", "p_subscribe")
ID_KEYS = ("customer_ids", "customer_id", "customers", "selected_customers", "selected_ids", "customer", "ids", "id")
MAX_SCAN_DEPTH = 3
SKIPPED_DIR_NAMES = {".git", ".ipynb_checkpoints", "sample_data", "__pycache__", "node_modules", "site-packages"}


def quick_tail(probabilities, threshold=SUCCESS_THRESHOLD):
    """P(K >= threshold) by dynamic programming (used only to screen candidate files)."""
    pmf = np.zeros(len(probabilities) + 1)
    pmf[0] = 1.0
    for p in probabilities:
        updated = pmf * (1.0 - p)
        updated[1:] += pmf[:-1] * p
        pmf = updated
    return float(pmf[threshold:].sum())


def is_number(value):
    return isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool)


def candidates_from_frame(frame, how):
    """(ids, probabilities, description) tuples from a table with id and probability columns."""
    lowered = {str(column).strip().lower(): column for column in frame.columns}
    id_column = next((lowered[key] for key in ID_KEYS if key in lowered), None)
    found = []
    for key in PROBABILITY_KEYS:
        if key not in lowered:
            continue
        column = lowered[key]
        values = pd.to_numeric(frame[column], errors="coerce")
        if values.isna().any():
            continue
        if id_column is not None:
            ids = frame[id_column].astype(str).str.strip().tolist()
            found.append((ids, values.tolist(), f"{how}, columns '{id_column}' and '{column}'"))
        elif len(frame) == NUM_CUSTOMERS:
            found.append((None, values.tolist(), f"{how}, column '{column}' (no customer-id column)"))
    return found


def candidates_from_json(content, how):
    found = []

    def walk(node, path):
        if isinstance(node, list):
            if node and all(is_number(x) for x in node):
                found.append((None, [float(x) for x in node], f"{how}, list at {path or '/'}"))
            elif node and all(isinstance(x, dict) for x in node):
                found.extend(candidates_from_frame(pd.DataFrame(node), f"{how}, records at {path or '/'}"))
            return
        if not isinstance(node, dict):
            return
        for key in PROBABILITY_KEYS:
            value = node.get(key)
            if isinstance(value, list) and value and all(is_number(x) for x in value):
                ids = next(([str(i) for i in node[k]] for k in ID_KEYS
                            if isinstance(node.get(k), list) and len(node[k]) == len(value)), None)
                found.append((ids, [float(x) for x in value], f"{how}, key '{key}' at {path or '/'}"))
        if node and all(is_number(v) for v in node.values()) and any(k in node for k in EXPECTED_CUSTOMER_IDS):
            found.append((list(node.keys()), [float(v) for v in node.values()], f"{how}, id->probability map at {path or '/'}"))
        for key, value in node.items():
            walk(value, f"{path}/{key}")

    walk(content, "")
    return found


def read_candidates(path):
    path = Path(path)
    how = path.name
    if path.suffix.lower() == ".json":
        with open(path) as handle:
            return candidates_from_json(json.load(handle), how)
    separator = "\t" if path.suffix.lower() == ".tsv" else ","
    return candidates_from_frame(pd.read_csv(path, sep=separator), how)


def resolve_candidate(ids, probabilities):
    """Resolve a candidate into the exact expected customer order.

    ID-bearing six-row inputs must match the expected set exactly. Larger
    customer tables may contain other unique IDs, from which the six expected
    records are selected. ID-less six-value inputs are simulation-only.
    """
    values = list(probabilities)
    if ids is None:
        if len(values) != NUM_CUSTOMERS:
            return None
        return None, values, "six values without customer IDs (simulation only; mapping unverified)"

    normalized_ids = [str(customer_id).strip() for customer_id in ids]
    if len(normalized_ids) != len(values) or any(not customer_id for customer_id in normalized_ids):
        return None
    if len(set(normalized_ids)) != len(normalized_ids):
        return None

    expected = list(EXPECTED_CUSTOMER_IDS)
    expected_set = set(expected)
    present = set(normalized_ids)
    if len(normalized_ids) == NUM_CUSTOMERS:
        if present != expected_set:
            return None
        rule = "mapped by customer ID into validated order"
    else:
        # A larger source table is acceptable only when it contains each
        # expected customer exactly once; unrelated rows are not selected.
        if len(normalized_ids) < NUM_CUSTOMERS or not expected_set.issubset(present):
            return None
        rule = "selected by customer ID from larger table into validated order"

    lookup = dict(zip(normalized_ids, values))
    return expected, [lookup[customer_id] for customer_id in expected], rule


def customer_ids_verified(ids):
    """True only for the exact expected customer sequence with verified IDs."""
    return ids is not None and list(ids) == list(EXPECTED_CUSTOMER_IDS)


def candidate_problems(ids, probabilities):
    """Reasons why a resolved candidate must be rejected (empty list = acceptable)."""
    problems = []
    values = np.array(probabilities, dtype=float)
    if len(values) != NUM_CUSTOMERS:
        problems.append(f"{len(values)} values, expected {NUM_CUSTOMERS}")
        return problems
    if not np.all(np.isfinite(values)) or np.any((values <= 0) | (values >= 1)):
        problems.append("a probability is not finite or not strictly between 0 and 1")
        return problems
    if ids is not None and list(ids) != list(EXPECTED_CUSTOMER_IDS):
        problems.append(f"customer IDs {ids} differ from the expected ordered IDs {EXPECTED_CUSTOMER_IDS}")
    difference = abs(quick_tail(values) - EXACT_REFERENCE_TAIL)
    if difference > REFERENCE_TOLERANCE:
        problems.append(f"P(K>=3) = {quick_tail(values):.16f} differs from the reference by {difference:.2e}")
    return problems


def list_data_files():
    files = []
    for root_text in STAGE3_SEARCH_DIRS:
        root = Path(root_text)
        if not root.is_dir():
            continue
        root_depth = len(root.resolve().parts)
        for directory, subdirectories, names in os.walk(root):
            here = Path(directory).resolve()
            subdirectories[:] = sorted(d for d in subdirectories if d not in SKIPPED_DIR_NAMES and not d.startswith("."))
            if here == OUTPUT_DIR.resolve() or OUTPUT_DIR.resolve() in here.parents:
                subdirectories[:] = []
                continue
            if len(here.parts) - root_depth >= MAX_SCAN_DEPTH:
                subdirectories[:] = []
            for name in sorted(names):
                candidate = here / name
                if candidate.suffix.lower() in (".json", ".csv", ".tsv") and candidate.stat().st_size < 50_000_000:
                    files.append(candidate)
            if len(files) > 3000:
                return files
    return sorted(set(files))


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def try_files(paths, rejection_log):
    for path in paths:
        try:
            candidates = read_candidates(path)
        except Exception as error:
            rejection_log.append(f"{path}: unreadable ({type(error).__name__})")
            continue
        for ids, probabilities, description in candidates:
            resolved = resolve_candidate(ids, probabilities)
            if resolved is None:
                continue
            chosen_ids, chosen_probabilities, rule = resolved
            problems = candidate_problems(chosen_ids, chosen_probabilities)
            if not problems:
                provenance = {"source_path": str(Path(path).resolve()), "sha256": sha256_of(path),
                              "size_bytes": Path(path).stat().st_size,
                              "modified_utc": datetime.fromtimestamp(Path(path).stat().st_mtime, timezone.utc).isoformat(timespec="seconds"),
                              "location_in_file": description, "selection_rule": rule,
                              "customer_ids_present_in_file": chosen_ids is not None}
                return chosen_ids, chosen_probabilities, provenance
            rejection_log.append(f"{path} [{description}]: " + "; ".join(problems))
    return None


def load_stage3_probabilities():
    rejection_log = []
    if MANUAL_PROBABILITIES is not None:
        ids = [str(i).strip() for i in MANUAL_CUSTOMER_IDS] if MANUAL_CUSTOMER_IDS is not None else None
        resolved = resolve_candidate(ids, [float(p) for p in MANUAL_PROBABILITIES])
        if resolved is None:
            raise ValueError("MANUAL_PROBABILITIES rejected: IDs must be unique and match the expected six; provide six values")
        chosen_ids, chosen_probabilities, rule = resolved
        problems = candidate_problems(chosen_ids, chosen_probabilities)
        if problems:
            raise ValueError("MANUAL_PROBABILITIES rejected: " + "; ".join(problems))
        return chosen_ids, chosen_probabilities, {"source_path": "MANUAL_PROBABILITIES (typed into the notebook)",
                                                  "selection_rule": rule, "customer_ids_present_in_file": chosen_ids is not None}
    paths = [Path(STAGE3_INPUT_FILE)] if STAGE3_INPUT_FILE else list_data_files()
    result = try_files(paths, rejection_log)
    if result is None:
        try:
            from google.colab import files as colab_files
        except ImportError:
            colab_files = None
        if colab_files is not None:
            print("No valid Stage 3 file found. Upload your project's Stage 3 probability file now.")
            uploaded = colab_files.upload()
            result = try_files([Path(name) for name in uploaded], rejection_log)
    if result is None:
        detail = "\n  ".join(rejection_log[:15]) if rejection_log else "(no candidate with six matching values was found)"
        raise FileNotFoundError(
            "STAGE 3 INPUT NOT FOUND. No file passed all checks (six probabilities in (0,1), expected customer IDs "
            f"{EXPECTED_CUSTOMER_IDS}, and P(K>=3) = {EXACT_REFERENCE_TAIL} within {REFERENCE_TOLERANCE}).\n"
            f"Searched {len(paths)} file(s) under {STAGE3_SEARCH_DIRS}.\nRejections:\n  {detail}\n"
            "Provide the project's validated Stage 3 file (full-precision probabilities) via STAGE3_INPUT_FILE or an upload. "
            "Nothing was substituted and no hardware experiment can run.")
    return result


loaded_ids, loaded_probabilities, PROBABILITY_PROVENANCE = load_stage3_probabilities()
stage3_probabilities = np.array(loaded_probabilities, dtype=float)
if loaded_ids is None:
    stage3_customer_ids = [f"position_{i}" for i in range(NUM_CUSTOMERS)]
    print("WARNING: the source file has no customer IDs. Values passed the reference check but IDs are NOT verified.")
else:
    stage3_customer_ids = list(loaded_ids)
PROBABILITY_PROVENANCE["loaded_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
PROBABILITY_PROVENANCE["customer_ids"] = stage3_customer_ids
PROBABILITIES_VERIFIED = customer_ids_verified(loaded_ids)
PROBABILITY_PROVENANCE["customer_ids_verified"] = PROBABILITIES_VERIFIED
PROBABILITY_PROVENANCE["simulation_only"] = not PROBABILITIES_VERIFIED
probability_source = PROBABILITY_PROVENANCE["source_path"]

print(f"Loaded from: {probability_source}")
print(f"  location: {PROBABILITY_PROVENANCE.get('location_in_file', 'n/a')} | rule: {PROBABILITY_PROVENANCE['selection_rule']}")
print(f"  sha256: {PROBABILITY_PROVENANCE.get('sha256', 'n/a')}")
print(pd.DataFrame({"qubit": range(NUM_CUSTOMERS), "customer": stage3_customer_ids,
                    "p_subscribe": stage3_probabilities}).to_string(index=False, float_format=lambda x: f"{x:.12f}"))
print(f"Customer-to-probability mapping verified: {PROBABILITIES_VERIFIED}")
print(f"P(K>=3) from these values: {quick_tail(stage3_probabilities):.16f} (reference {EXACT_REFERENCE_TAIL:.16f})")

## 7. Exact mathematical reference calculations
$K=\sum_i X_i$ with independent $X_i\sim\mathrm{Bernoulli}(p_i)$ follows a Poisson-binomial distribution. Two independent exact methods:

1. **Dynamic programming.** With $f_m(k)=P(K_m=k)$ after $m$ customers: $f_{m}(k)=f_{m-1}(k)(1-p_m)+f_{m-1}(k-1)\,p_m$.
2. **Enumeration** of all $2^6=64$ outcomes, $P(x)=\prod_i p_i^{x_i}(1-p_i)^{1-x_i}$.

**Bit-ordering convention used everywhere:** outcome integer $x=\sum_i x_i 2^i$, i.e. *qubit $i$ is bit $i$ (least significant = qubit 0)*. Qiskit prints bitstrings as `q5 q4 ... q0`, which is the same convention.

In [ ]:
def poisson_binomial_pmf(probabilities):
    """P(K = k) for k = 0..n by dynamic programming."""
    pmf = np.zeros(len(probabilities) + 1)
    pmf[0] = 1.0
    for p in probabilities:
        updated = pmf * (1.0 - p)
        updated[1:] += pmf[:-1] * p
        pmf = updated
    return pmf


def enumerate_outcome_probabilities(probabilities):
    """Probability of each of the 2^n outcomes; index bit i = qubit i."""
    n = len(probabilities)
    outcome_probabilities = np.zeros(2 ** n)
    for outcome in range(2 ** n):
        probability = 1.0
        for qubit, p in enumerate(probabilities):
            probability *= p if (outcome >> qubit) & 1 else 1.0 - p
        outcome_probabilities[outcome] = probability
    return outcome_probabilities


POPULATION_COUNT = np.array([bin(x).count("1") for x in range(2 ** NUM_CUSTOMERS)])


def tail_from_pmf(pmf, threshold=SUCCESS_THRESHOLD):
    return float(pmf[threshold:].sum())


def pmf_from_outcome_probabilities(outcome_probabilities):
    """Collapse 64 outcome probabilities into P(K = k)."""
    return np.bincount(POPULATION_COUNT, weights=outcome_probabilities, minlength=NUM_CUSTOMERS + 1)


exact_pmf_dp = poisson_binomial_pmf(stage3_probabilities)
exact_outcomes = enumerate_outcome_probabilities(stage3_probabilities)
exact_pmf_enumeration = pmf_from_outcome_probabilities(exact_outcomes)

tail_dp = tail_from_pmf(exact_pmf_dp)
tail_enumeration = tail_from_pmf(exact_pmf_enumeration)

print(f"P(K>=3) dynamic programming : {tail_dp:.16f}")
print(f"P(K>=3) enumeration (64)    : {tail_enumeration:.16f}")
print(f"P(K>=3) validated reference : {EXACT_REFERENCE_TAIL:.16f}")

assert abs(exact_outcomes.sum() - 1.0) < 1e-12, "Outcome probabilities do not sum to 1."
assert abs(exact_pmf_dp.sum() - 1.0) < 1e-12, "Poisson-binomial pmf does not sum to 1."
assert abs(tail_dp - tail_enumeration) < 1e-12, "DP and enumeration disagree."
if abs(tail_dp - EXACT_REFERENCE_TAIL) > REFERENCE_TOLERANCE:
    raise ValueError(
        f"The loaded probabilities give P(K>=3)={tail_dp:.16f}, which differs from the validated "
        f"reference {EXACT_REFERENCE_TAIL:.16f}. These are not the Stage 3 probabilities; "
        "check the input file / customer order before continuing.")
print("Reference check passed: the loaded probabilities reproduce the validated Stage 3 value.")

# The reference value is used only for comparison; it is never altered.
EXACT_TAIL = tail_dp
print("\nExact distribution of K:")
print(pd.DataFrame({"k": range(NUM_CUSTOMERS + 1), "P(K=k)": exact_pmf_dp}).to_string(index=False))

## 8. Quantum circuit construction
### Experiment A — independent-Bernoulli circuit (real Stage 3 data)
One $R_Y(2\arcsin\sqrt{p_i})$ per qubit, then measurement of all qubits.

### Experiment B — exact-cardinality optimisation (**SYNTHETIC instance, not Atomiq data**)
The deep ten-qubit Stage 2 circuit is **not** sent to hardware (depth about 6,010). The actual ten-customer Stage 2 problem stays a classical exact result. Experiment B studies a **synthetic** six-candidate instance with exact cardinality $k=3$:
$$\max_{x\in\{0,1\}^6,\ \sum x_i=3}\ \sum_i v_i x_i-\sum_{i=0}^{4} w_i\,x_i x_{i+1}.$$
Values $v_i$ and neighbour penalties $w_i$ are seeded random numbers. The interaction graph is a path, so the circuit maps onto hardware chains without SWAP routing. Results are a hardware-sized *surrogate* and must never be reported as results on the challenge data.

QAOA minimises $C(x)=-\text{reward}(x)$ with a **path XY mixer**, which conserves Hamming weight.

* Initial state $\lvert 010101\rangle$ (weight 3, feasible).
* Cost layer $e^{-i\gamma C}$. With $x_i=(1-z_i)/2$: $C=c_0+\sum_i h_i Z_i+\sum_i J_i Z_iZ_{i+1}$, implemented as $R_Z(2\gamma h_i)=e^{-i\gamma h_i Z}$ and $R_{ZZ}(2\gamma J_i)=e^{-i\gamma J_i ZZ}$.
* Mixer. Define $H_{ab}=X_aX_b+Y_aY_b$ for neighbours $a,b$. Each pair gate is $e^{-i\beta H_{ab}}=R_{XX}(2\beta)R_{YY}(2\beta)$ (exact, because $XX$ and $YY$ commute), also available as one `XXPlusYYGate(4β)`. Even pairs $(0,1),(2,3),(4,5)$ are applied first, then odd pairs $(1,2),(3,4)$.
* **Important:** pairs that share a qubit do not commute, so this sequential layer is a first-order **product-formula (Trotter-type) approximation** of the global path-XY evolution $e^{-i\beta\sum_{\text{edges}}H_{ab}}$, not an exact copy of it. It is a legitimate variational ansatz because every pair gate commutes with the total $Z$, so Hamming weight is conserved *exactly*. Section 9b verifies all of this numerically.

In [ ]:
def build_bernoulli_circuit(probabilities, measure=True):
    """Experiment A: qubit i is prepared in sqrt(1-p_i)|0> + sqrt(p_i)|1>."""
    circuit = QuantumCircuit(len(probabilities), name="stage3_bernoulli")
    for qubit, p in enumerate(probabilities):
        circuit.ry(2.0 * math.asin(math.sqrt(p)), qubit)
    if measure:
        circuit.measure_all()          # classical register is named "meas"
    return circuit


def build_readout_calibration_circuits(num_qubits):
    """Prepare |0...0> and |1...1> and measure: used for readout-error calibration."""
    all_zero = QuantumCircuit(num_qubits, name="readout_all_zero")
    all_zero.measure_all()
    all_one = QuantumCircuit(num_qubits, name="readout_all_one")
    all_one.x(range(num_qubits))
    all_one.measure_all()
    return all_zero, all_one


@dataclass
class SyntheticSelectionProblem:
    """Synthetic exact-cardinality selection problem on a path graph. NOT Atomiq data."""
    values: np.ndarray            # v_i
    neighbour_penalties: np.ndarray   # w_i between candidate i and i+1
    select_count: int

    @property
    def size(self):
        return len(self.values)


def make_synthetic_problem(seed, size=NUM_CUSTOMERS, select_count=SELECT_COUNT):
    generator = np.random.default_rng(seed)
    values = np.round(generator.uniform(1.0, 2.0, size), 2)
    penalties = np.round(generator.uniform(0.3, 1.2, size - 1), 2)
    return SyntheticSelectionProblem(values, penalties, select_count)


def cost_of_outcome(problem, outcome):
    """C(x) = -reward(x); bit i of `outcome` is x_i."""
    bits = [(outcome >> i) & 1 for i in range(problem.size)]
    reward = sum(problem.values[i] * bits[i] for i in range(problem.size))
    reward -= sum(problem.neighbour_penalties[i] * bits[i] * bits[i + 1] for i in range(problem.size - 1))
    return -reward


def ising_coefficients(problem):
    """C(x) = constant + sum_i h_i z_i + sum_i J_i z_i z_{i+1}, with z_i = 1 - 2 x_i."""
    n = problem.size
    linear = problem.values / 2.0
    linear = linear.copy()
    for i, w in enumerate(problem.neighbour_penalties):
        linear[i] -= w / 4.0
        linear[i + 1] -= w / 4.0
    quadratic = problem.neighbour_penalties / 4.0
    constant = -problem.values.sum() / 2.0 + problem.neighbour_penalties.sum() / 4.0
    return linear, quadratic, constant


def append_xy_pair(circuit, beta, a, b, native_xy=True):
    """exp(-i beta (X_a X_b + Y_a Y_b)).  XXPlusYYGate(theta, 0) = exp(-i theta/4 (XX+YY)), so theta = 4 beta."""
    if native_xy:
        circuit.append(XXPlusYYGate(4.0 * beta), [a, b])
    else:
        circuit.rxx(2.0 * beta, a, b)      # RXX(t) = exp(-i t/2 XX)
        circuit.ryy(2.0 * beta, a, b)      # RYY(t) = exp(-i t/2 YY); XX and YY commute, so the product is exact


def append_mixer_layer(circuit, beta, native_xy=True):
    """One sequential path-XY mixer layer: even edges first, then odd edges (Trotter-style product)."""
    n = circuit.num_qubits
    for a, b in [(i, i + 1) for i in range(0, n - 1, 2)] + [(i, i + 1) for i in range(1, n - 1, 2)]:
        append_xy_pair(circuit, beta, a, b, native_xy)


def append_cost_layer(circuit, gamma, linear, quadratic):
    for qubit, h in enumerate(linear):
        circuit.rz(2.0 * gamma * h, qubit)                     # RZ(t) = exp(-i t/2 Z)  -> exp(-i gamma h Z)
    for i, j in enumerate(quadratic):
        circuit.rzz(2.0 * gamma * j, i, i + 1)                 # RZZ(t) = exp(-i t/2 ZZ) -> exp(-i gamma J ZZ)


def build_xy_qaoa_circuit(problem, gammas, betas, measure=True, native_xy=True):
    """Path-XY-mixer QAOA with weight-preserving mixer; starts in a feasible basis state."""
    n, k = problem.size, problem.select_count
    if len(gammas) != len(betas):
        raise ValueError("gammas and betas must have the same length.")
    if 2 * k > n:
        raise ValueError("The alternating initial state needs 2*k <= n.")
    linear, quadratic, _ = ising_coefficients(problem)
    circuit = QuantumCircuit(n, name=f"xy_qaoa_p{len(gammas)}")
    for qubit in range(0, 2 * k, 2):                   # |010101> for n=6, k=3 (qubits 0, 2, 4 set)
        circuit.x(qubit)
    for gamma, beta in zip(gammas, betas):
        append_cost_layer(circuit, gamma, linear, quadratic)
        append_mixer_layer(circuit, beta, native_xy)
    if measure:
        circuit.measure_all()
    return circuit


problem = make_synthetic_problem(SYNTHETIC_PROBLEM_SEED)
outcome_costs = np.array([cost_of_outcome(problem, x) for x in range(2 ** problem.size)])
feasible_mask = POPULATION_COUNT == problem.select_count
feasible_outcomes = np.flatnonzero(feasible_mask)
optimal_outcome = int(feasible_outcomes[np.argmin(outcome_costs[feasible_outcomes])])
optimal_cost = float(outcome_costs[optimal_outcome])
worst_feasible_cost = float(outcome_costs[feasible_outcomes].max())

greedy_outcome = int(sum(1 << i for i in np.argsort(-problem.values)[: problem.select_count]))
format_selection = lambda x: [i for i in range(problem.size) if (x >> i) & 1]

print("SYNTHETIC instance (not Atomiq customer data)")
print(f"  values v          : {problem.values}")
print(f"  neighbour penalty w: {problem.neighbour_penalties}")
print(f"  feasible solutions: {len(feasible_outcomes)}")
print(f"  exact optimum     : items {format_selection(optimal_outcome)}, reward {-optimal_cost:.3f}")
print(f"  greedy top-3 value: items {format_selection(greedy_outcome)}, reward {-outcome_costs[greedy_outcome]:.3f}")

## 9. Circuit validation and Aer simulation
Every check below must pass before any hardware job is allowed. Checks cover: probability normalisation, the Ising mapping, qubit/bit ordering, measurement interpretation, weight conservation of the XY mixer, and agreement between ideal simulation and exact values.

In [ ]:
VALIDATION_LOG = []


def check(description, condition, detail=""):
    """Record one validation check; raise immediately if it fails."""
    VALIDATION_LOG.append({"check": description, "passed": bool(condition), "detail": detail})
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {description} {detail}")
    if not condition:
        raise AssertionError(f"Validation failed: {description} {detail}")


def counts_to_probability_vector(counts, num_qubits):
    """Convert Qiskit counts (bitstring 'q_{n-1}...q_0') to a vector indexed by the integer."""
    shots = sum(counts.values())
    vector = np.zeros(2 ** num_qubits)
    for bitstring, count in counts.items():
        vector[int(bitstring.replace(" ", ""), 2)] += count
    return vector / shots


def tail_from_counts(counts, threshold=SUCCESS_THRESHOLD):
    shots = sum(counts.values())
    successes = sum(c for bits, c in counts.items() if bits.replace(" ", "").count("1") >= threshold)
    return successes / shots


# ---- Experiment A circuit ----------------------------------------------------
circuit_a = build_bernoulli_circuit(stage3_probabilities)
state_a = Statevector(build_bernoulli_circuit(stage3_probabilities, measure=False))
statevector_outcomes = state_a.probabilities()
check("A: statevector probabilities sum to 1", abs(statevector_outcomes.sum() - 1) < 1e-12)
check("A: statevector equals the product distribution (qubit i = bit i)",
      np.allclose(statevector_outcomes, exact_outcomes, atol=1e-12),
      f"max |diff| = {np.abs(statevector_outcomes - exact_outcomes).max():.1e}")
check("A: exact P(K>=3) from the circuit statevector matches the reference",
      abs(tail_from_pmf(pmf_from_outcome_probabilities(statevector_outcomes)) - EXACT_TAIL) < 1e-12)

# Ordering self-test with a deliberately asymmetric circuit: flip qubit 0 only.
ordering_circuit = QuantumCircuit(NUM_CUSTOMERS)
ordering_circuit.x(0)
ordering_circuit.measure_all()
ordering_counts = AerSimulator().run(ordering_circuit, shots=64, seed_simulator=RANDOM_SEED).result().get_counts()
check("Ordering: X on qubit 0 gives bitstring '000001' (qubit 0 is the rightmost bit)",
      list(ordering_counts) == ["000001"], str(ordering_counts))

# ---- Experiment B circuit ----------------------------------------------------
ising_linear, ising_quadratic, ising_constant = ising_coefficients(problem)
reconstructed = []
for x in range(2 ** problem.size):
    z = [1 - 2 * ((x >> i) & 1) for i in range(problem.size)]
    reconstructed.append(ising_constant + sum(ising_linear[i] * z[i] for i in range(problem.size))
                         + sum(ising_quadratic[i] * z[i] * z[i + 1] for i in range(problem.size - 1)))
check("B: Ising form reproduces the cost of all 64 bitstrings",
      np.allclose(reconstructed, outcome_costs, atol=1e-12),
      f"max |diff| = {np.abs(np.array(reconstructed) - outcome_costs).max():.1e}")

test_circuit = build_xy_qaoa_circuit(problem, [0.37, 0.21], [0.41, 0.13], measure=False)
test_probabilities = Statevector(test_circuit).probabilities()
check("B: XY-mixer QAOA keeps all probability on weight-3 states",
      abs(test_probabilities[feasible_mask].sum() - 1.0) < 1e-10,
      f"feasible mass = {test_probabilities[feasible_mask].sum():.12f}")
check("B: statevector probabilities are normalised", abs(test_probabilities.sum() - 1) < 1e-12)

In [ ]:
def expected_cost_ideal(parameters, layers):
    gammas, betas = parameters[:layers], parameters[layers:]
    circuit = build_xy_qaoa_circuit(problem, gammas, betas, measure=False)
    return float(Statevector(circuit).probabilities() @ outcome_costs)


def optimise_qaoa_parameters(layers, num_starts=12, seed=RANDOM_SEED):
    """Noise-free multi-start Nelder-Mead on the exact expected cost (runs locally, costs no QPU time)."""
    generator = np.random.default_rng(seed + layers)
    best = None
    for _ in range(num_starts):
        start = np.concatenate([generator.uniform(0, np.pi, layers), generator.uniform(0, np.pi / 2, layers)])
        result = optimize.minimize(expected_cost_ideal, start, args=(layers,), method="Nelder-Mead",
                                   options={"xatol": 1e-4, "fatol": 1e-6, "maxiter": 400})
        if best is None or result.fun < best.fun:
            best = result
    return best.x[:layers], best.x[layers:], float(best.fun)


def selection_metrics(outcome_probabilities):
    """Quality of a distribution over bitstrings for the selection problem (see section 13)."""
    feasibility = float(outcome_probabilities[feasible_mask].sum())
    metrics = {"feasibility": feasibility,
               "optimal_probability": float(outcome_probabilities[optimal_outcome])}
    if feasibility > 0:
        conditional = outcome_probabilities * feasible_mask / feasibility
        mean_cost = float(conditional @ outcome_costs)
        metrics.update({
            "optimal_probability_given_feasible": float(conditional[optimal_outcome]),
            "mean_feasible_cost": mean_cost,
            "regret": mean_cost - optimal_cost,          # expected reward lost vs the optimum
            "approximation_ratio": (worst_feasible_cost - mean_cost) / (worst_feasible_cost - optimal_cost)})
    else:
        metrics.update({"optimal_probability_given_feasible": float("nan"), "mean_feasible_cost": float("nan"),
                        "regret": float("nan"), "approximation_ratio": float("nan")})
    return metrics


qaoa_ideal_results = {}
for layers in (1, 2):
    gammas, betas, best_cost = optimise_qaoa_parameters(layers)
    probabilities = Statevector(build_xy_qaoa_circuit(problem, gammas, betas, measure=False)).probabilities()
    qaoa_ideal_results[layers] = {"gammas": gammas, "betas": betas, "probabilities": probabilities,
                                  **selection_metrics(probabilities)}
    print(f"p={layers}: gammas={np.round(gammas, 3)}, betas={np.round(betas, 3)}, "
          f"P(optimum)={qaoa_ideal_results[layers]['optimal_probability']:.3f}, "
          f"approx ratio={qaoa_ideal_results[layers]['approximation_ratio']:.3f}")

# Reference baselines for Experiment B (no quantum circuit involved).
uniform_all = np.full(2 ** problem.size, 1.0 / 2 ** problem.size)
uniform_feasible = feasible_mask / feasible_mask.sum()
baseline_metrics = {"random bitstring": selection_metrics(uniform_all),
                    "random feasible": selection_metrics(uniform_feasible)}

qaoa_gammas = qaoa_ideal_results[QAOA_LAYERS]["gammas"]
qaoa_betas = qaoa_ideal_results[QAOA_LAYERS]["betas"]
circuit_b = build_xy_qaoa_circuit(problem, qaoa_gammas, qaoa_betas)
print(f"\nExperiment B uses p={QAOA_LAYERS}; logical gate counts: {dict(circuit_b.count_ops())}")

In [ ]:
# ---- Ideal (noise-free) Aer sampling for both experiments ----------------------
ideal_simulator = AerSimulator()


def sample_ideal(circuit, shots, seed):
    return ideal_simulator.run(transpile(circuit, ideal_simulator, seed_transpiler=RANDOM_SEED), shots=shots, seed_simulator=seed).result().get_counts()


ideal_counts_a = sample_ideal(circuit_a, SHOTS_STAGE3, RANDOM_SEED)
ideal_tail_a = tail_from_counts(ideal_counts_a)
check("A: ideal Aer sampling agrees with exact P(K>=3) within 4 binomial standard errors",
      abs(ideal_tail_a - EXACT_TAIL) < 4 * math.sqrt(EXACT_TAIL * (1 - EXACT_TAIL) / SHOTS_STAGE3),
      f"estimate {ideal_tail_a:.4f} vs exact {EXACT_TAIL:.4f}")

ideal_counts_b = sample_ideal(circuit_b, SHOTS_QAOA, RANDOM_SEED)
ideal_vector_b = counts_to_probability_vector(ideal_counts_b, problem.size)
check("B: ideal Aer sampling returns only feasible bitstrings", ideal_vector_b[~feasible_mask].sum() == 0)
check("B: ideal Aer sampling agrees with the statevector optimum probability (4 sigma)",
      abs(ideal_vector_b[optimal_outcome] - qaoa_ideal_results[QAOA_LAYERS]["optimal_probability"])
      < 4 * math.sqrt(0.25 / SHOTS_QAOA))

IDEAL_QAOA_DEPTH_COUNTS = {}
for layers in (1, 2):
    depth_circuit = build_xy_qaoa_circuit(problem, qaoa_ideal_results[layers]["gammas"],
                                          qaoa_ideal_results[layers]["betas"], measure=True)
    depth_counts = sample_ideal(depth_circuit, SHOTS_QAOA_HARDWARE, RANDOM_SEED + 100 + layers)
    depth_label = f"B_p{layers}_best"
    IDEAL_QAOA_DEPTH_COUNTS[depth_label] = depth_counts
    depth_vector = counts_to_probability_vector(depth_counts, problem.size)
    check(f"B p={layers}: ideal high-shot sampler preserves exact cardinality",
          depth_vector[~feasible_mask].sum() == 0)

VALIDATION_PASSED = all(entry["passed"] for entry in VALIDATION_LOG)
print(f"\nAll {len(VALIDATION_LOG)} validation checks passed: {VALIDATION_PASSED}")

## 9b. Rigorous check of the Stage 2 XY-QAOA construction (blocks hardware if anything fails)
Checks, all numerical and independent of the circuit builder where possible:

1. The pair gate $e^{-i\beta(XX+YY)}$ built as `RXX·RYY` and as `XXPlusYYGate(4β)` both equal the matrix exponential.
2. The initial state has exactly weight $k=3$.
3. The cost layer is diagonal with phases $e^{-i\gamma C(x)}$ (sign and angle convention of `RZ`/`RZZ`).
4. A full mixer layer has zero matrix elements between states of different Hamming weight.
5. The sequential mixer is **not** the exact global XY evolution. We measure the gap and its scaling with $\beta$ (about $\beta^2$ for the first-order product formula).

In [ ]:
def pauli_matrix_xy(n, pairs):
    """Dense matrix of sum over (a,b) of X_a X_b + Y_a Y_b (qubit i = bit i)."""
    terms = []
    for a, b in pairs:
        terms += [("XX", [a, b], 1.0), ("YY", [a, b], 1.0)]
    return SparsePauliOp.from_sparse_list(terms, num_qubits=n).to_matrix()


path_pairs = [(i, i + 1) for i in range(problem.size - 1)]
beta_check = 0.37
pair_exact = expm(-1j * beta_check * pauli_matrix_xy(2, [(0, 1)]))
for native in (True, False):
    pair_circuit = QuantumCircuit(2)
    append_xy_pair(pair_circuit, beta_check, 0, 1, native_xy=native)
    check(f"B: pair gate exp(-i beta (XX+YY)) equals the matrix exponential (native_xy={native}, exact incl. phase)",
          np.allclose(Operator(pair_circuit).data, pair_exact, atol=1e-12),
          f"max |diff| = {np.abs(Operator(pair_circuit).data - pair_exact).max():.1e}")

init_circuit = build_xy_qaoa_circuit(problem, [], [], measure=False)
init_probabilities = Statevector(init_circuit).probabilities()
check("B: initial state is a single basis state of Hamming weight k",
      np.count_nonzero(init_probabilities > 1e-12) == 1 and POPULATION_COUNT[np.argmax(init_probabilities)] == problem.select_count,
      f"state = {format(int(np.argmax(init_probabilities)), '06b')}")

gamma_check = 0.29
linear_check, quadratic_check, _ = ising_coefficients(problem)
cost_circuit = QuantumCircuit(problem.size)
append_cost_layer(cost_circuit, gamma_check, linear_check, quadratic_check)
cost_diagonal = np.diag(Operator(cost_circuit).data)
check("B: cost layer is diagonal", np.allclose(Operator(cost_circuit).data, np.diag(cost_diagonal), atol=1e-12))
expected_phase = np.exp(-1j * gamma_check * (outcome_costs - outcome_costs[0]))
check("B: cost-layer phases equal exp(-i gamma (C(x) - C(0))) for all 64 states (sign and angle convention)",
      np.allclose(cost_diagonal / cost_diagonal[0], expected_phase, atol=1e-10),
      f"max |diff| = {np.abs(cost_diagonal / cost_diagonal[0] - expected_phase).max():.1e}")

same_weight = POPULATION_COUNT[:, None] == POPULATION_COUNT[None, :]
for native in (True, False):
    mixer_circuit = QuantumCircuit(problem.size)
    append_mixer_layer(mixer_circuit, 0.61, native_xy=native)
    mixer_matrix = Operator(mixer_circuit).data
    leakage = np.abs(mixer_matrix[~same_weight]).max()
    check(f"B: mixer layer has no matrix element between different Hamming weights (native_xy={native})",
          leakage < 1e-12, f"max leakage amplitude = {leakage:.1e}")

full_unitary_native = Operator(build_xy_qaoa_circuit(problem, qaoa_ideal_results[1]["gammas"], qaoa_ideal_results[1]["betas"],
                                                     measure=False, native_xy=True)).data
full_unitary_rxx = Operator(build_xy_qaoa_circuit(problem, qaoa_ideal_results[1]["gammas"], qaoa_ideal_results[1]["betas"],
                                                  measure=False, native_xy=False)).data
check("B: the XXPlusYY and RXX/RYY circuit variants are the same unitary",
      np.allclose(full_unitary_native, full_unitary_rxx, atol=1e-10))

print("\nProduct-formula gap: sequential even/odd mixer vs exact global path-XY evolution exp(-i beta sum H_ab)")
global_hamiltonian = pauli_matrix_xy(problem.size, path_pairs)
gap_rows = []
for beta_value in (0.4, 0.2, 0.1, 0.05):
    sequential = QuantumCircuit(problem.size)
    append_mixer_layer(sequential, beta_value)
    gap = np.linalg.norm(Operator(sequential).data - expm(-1j * beta_value * global_hamiltonian), 2)
    gap_rows.append({"beta": beta_value, "operator_norm_gap": gap})
gap_table = pd.DataFrame(gap_rows)
gap_table["ratio_to_previous"] = gap_table.operator_norm_gap.shift(1) / gap_table.operator_norm_gap
print(gap_table.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
check("B: sequential mixer differs from the exact global evolution and the gap shrinks about 4x per halving of beta (first-order product formula)",
      gap_table.operator_norm_gap.iloc[0] > 1e-3 and 3.0 < gap_table.ratio_to_previous.iloc[-1] < 5.0)
VALIDATION_PASSED = all(entry["passed"] for entry in VALIDATION_LOG)
print(f"\nAll {len(VALIDATION_LOG)} validation checks passed so far: {VALIDATION_PASSED}")

## 10. Transpilation and hardware compatibility checks
### Physical qubit selection (also the basis of Experiment C)
* **Experiment A** has no two-qubit gates, so connectivity is irrelevant and the mapping only matters through readout and single-qubit errors. We compare three groups of six qubits: *best*, *middle* (median quality) and *poor* (90th percentile of error, but still usable).
* **Experiment B** needs a chain of six coupled qubits. We enumerate all simple paths of length six in the coupling graph, score each by its summed two-qubit and readout errors, and compare the *best* path with a *typical* (median-score) path. Both need zero SWAPs, so differences come from calibration quality alone. All other settings (transpiler seed, optimisation level, shots) are identical, which keeps the comparison fair.

In [ ]:
def choose_single_qubit_layouts(calibration):
    """Three sets of six qubits ranked by readout + sx error."""
    usable = calibration.dropna(subset=["readout_error", "sx_error"])
    usable = usable[usable.readout_error < 0.5].copy()
    usable["score"] = usable.readout_error + usable.sx_error
    ranked = usable.sort_values("score").reset_index(drop=True)
    middle_start = len(ranked) // 2 - NUM_CUSTOMERS // 2
    poor_start = int(0.9 * len(ranked))
    return {"best": ranked.qubit[:NUM_CUSTOMERS].tolist(),
            "middle": ranked.qubit[middle_start:middle_start + NUM_CUSTOMERS].tolist(),
            "poor": ranked.qubit[poor_start:poor_start + NUM_CUSTOMERS].tolist()}


def enumerate_paths(edge_table, usable_qubits, length):
    """All simple paths with `length` qubits in the (undirected) coupling graph."""
    neighbours = {}
    for row in edge_table.itertuples():
        if row.qubit_a in usable_qubits and row.qubit_b in usable_qubits and not math.isnan(row.error):
            neighbours.setdefault(row.qubit_a, set()).add(row.qubit_b)
            neighbours.setdefault(row.qubit_b, set()).add(row.qubit_a)
    paths = []

    def extend(path):
        if len(path) == length:
            if path[0] < path[-1]:          # keep one direction of each path
                paths.append(list(path))
            return
        for nxt in neighbours.get(path[-1], ()):
            if nxt not in path:
                path.append(nxt)
                extend(path)
                path.pop()

    for start in neighbours:
        extend([start])
    return paths


def choose_chain_layouts(calibration, edge_table):
    readout = calibration.set_index("qubit").readout_error
    usable = set(calibration[calibration.readout_error < 0.5].qubit)
    edge_error = {}
    for row in edge_table.itertuples():
        edge_error[(row.qubit_a, row.qubit_b)] = edge_error[(row.qubit_b, row.qubit_a)] = row.error
    candidates = enumerate_paths(edge_table, usable, NUM_CUSTOMERS)
    if not candidates:
        raise RuntimeError("No coupled six-qubit path found on this backend.")
    scored = []
    for path in candidates:
        two_qubit = sum(edge_error[(a, b)] for a, b in zip(path[:-1], path[1:]))
        scored.append((two_qubit + sum(readout[q] for q in path), path))
    scored.sort(key=lambda item: item[0])
    return {"best": scored[0][1], "typical": scored[len(scored) // 2][1]}, scored


single_qubit_layouts = choose_single_qubit_layouts(qubit_calibration)
chain_layouts, all_chain_scores = choose_chain_layouts(qubit_calibration, edge_calibration)
print("Experiment A layouts:", single_qubit_layouts)
print("Experiment B layouts:", chain_layouts, f"({len(all_chain_scores)} candidate paths)")

In [ ]:
@dataclass
class PreparedCircuit:
    label: str
    experiment: str
    logical_circuit: QuantumCircuit
    isa_circuit: QuantumCircuit
    layout_name: str
    metrics: dict = field(default_factory=dict)


def estimate_success_probability(isa_circuit, target):
    """Product of (1 - gate error) over all gates and measurements (an 'ESP'-style estimate)."""
    success = 1.0
    for instruction in isa_circuit.data:
        name = instruction.operation.name
        if name in ("barrier", "delay") or name not in target.operation_names:
            continue
        qubits = tuple(isa_circuit.find_bit(q).index for q in instruction.qubits)
        properties = target[name].get(qubits)
        if properties is not None and properties.error is not None:
            success *= 1.0 - properties.error
    return success


def check_hardware_compatibility(isa_circuit, backend, physical_qubits):
    """Every instruction must be native on exactly the qubits it is applied to."""
    target = backend.target
    for instruction in isa_circuit.data:
        name = instruction.operation.name
        if name == "barrier":
            continue
        qubits = tuple(isa_circuit.find_bit(q).index for q in instruction.qubits)
        if not target.instruction_supported(name, qubits):
            raise RuntimeError(f"Instruction {name} on physical qubits {qubits} is not supported by {backend.name}.")
    active = {isa_circuit.find_bit(q).index for inst in isa_circuit.data
              if inst.operation.name != "barrier" for q in inst.qubits}
    if not active.issubset(set(physical_qubits)):
        raise RuntimeError(f"Circuit uses qubits {sorted(active - set(physical_qubits))} outside the chosen layout "
                           "(unexpected SWAP routing).")


def prepare_circuit(label, experiment, circuit, physical_qubits, layout_name):
    """Transpile with a fixed layout, verify compatibility, and record metrics."""
    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=3,
                                                initial_layout=list(physical_qubits),
                                                seed_transpiler=RANDOM_SEED)
    isa_circuit = pass_manager.run(circuit)
    check_hardware_compatibility(isa_circuit, backend, physical_qubits)
    operation_counts = dict(isa_circuit.count_ops())
    try:
        duration = float(isa_circuit.estimate_duration(backend.target, unit="s"))
    except Exception:
        duration = float("nan")
    metrics = {
        "physical_qubits": [int(q) for q in physical_qubits],
        "native_checked": True,
        "depth": isa_circuit.depth(),
        "two_qubit_depth": isa_circuit.depth(lambda instruction: len(instruction.qubits) > 1),
        "two_qubit_gates": isa_circuit.num_nonlocal_gates(),
        "total_gates": sum(v for k, v in operation_counts.items() if k not in ("measure", "barrier", "delay")),
        "gate_counts": operation_counts,
        "logical_depth": circuit.depth(),
        "logical_two_qubit_gates": circuit.num_nonlocal_gates(),
        "estimated_duration_s": duration,
        "estimated_success_probability": estimate_success_probability(isa_circuit, backend.target),
    }
    return PreparedCircuit(label, experiment, circuit, isa_circuit, layout_name, metrics)


PREPARED = {}
calibration_zero, calibration_one = build_readout_calibration_circuits(NUM_CUSTOMERS)

for layout_name, qubits in single_qubit_layouts.items():
    PREPARED[f"A_{layout_name}"] = prepare_circuit(f"A_{layout_name}", "A", circuit_a, qubits, layout_name)
    PREPARED[f"cal0_A_{layout_name}"] = prepare_circuit(f"cal0_A_{layout_name}", "calibration", calibration_zero, qubits, layout_name)
    PREPARED[f"cal1_A_{layout_name}"] = prepare_circuit(f"cal1_A_{layout_name}", "calibration", calibration_one, qubits, layout_name)
for layout_name, qubits in chain_layouts.items():
    PREPARED[f"B_{layout_name}"] = prepare_circuit(f"B_{layout_name}", "B", circuit_b, qubits, layout_name)
    PREPARED[f"cal0_B_{layout_name}"] = prepare_circuit(f"cal0_B_{layout_name}", "calibration", calibration_zero, qubits, layout_name)
    PREPARED[f"cal1_B_{layout_name}"] = prepare_circuit(f"cal1_B_{layout_name}", "calibration", calibration_one, qubits, layout_name)

metrics_table = pd.DataFrame([
    {"circuit": p.label, "experiment": p.experiment, "layout": p.layout_name,
     "physical_qubits": p.metrics["physical_qubits"], "depth": p.metrics["depth"],
     "two_qubit_gates": p.metrics["two_qubit_gates"], "total_gates": p.metrics["total_gates"],
     "logical_depth": p.metrics["logical_depth"], "logical_2q": p.metrics["logical_two_qubit_gates"],
     "duration_us": p.metrics["estimated_duration_s"] * 1e6,
     "est_success_prob": p.metrics["estimated_success_probability"]}
    for p in PREPARED.values() if p.experiment != "calibration"])
print(metrics_table.to_string(index=False))
check("Transpiled circuits are native on the backend and use only the chosen physical qubits",
      all(p.metrics.get("native_checked") for p in PREPARED.values()))
check("Experiment A uses no two-qubit gates after transpilation",
      all(PREPARED[f"A_{name}"].metrics["two_qubit_gates"] == 0 for name in single_qubit_layouts))
VALIDATION_PASSED = all(entry["passed"] for entry in VALIDATION_LOG)

### Decomposition and routing evidence for Experiment B
We transpile the same QAOA circuit with both mixer constructions on the same chain and compare what actually reaches the hardware. Nothing is called "hardware-efficient" here; only the measured counts below are reported, and the variant with fewer two-qubit gates (then lower depth) is the one used for hardware.

In [ ]:
def swap_count(isa_circuit):
    return int(isa_circuit.count_ops().get("swap", 0))


variant_rows, variant_prepared = [], {}
for variant_name, native in (("XXPlusYYGate", True), ("RXX + RYY", False)):
    variant_circuit = build_xy_qaoa_circuit(problem, qaoa_gammas, qaoa_betas, native_xy=native)
    prepared_variant = prepare_circuit(f"B_best[{variant_name}]", "B", variant_circuit, chain_layouts["best"], "best")
    variant_prepared[variant_name] = (variant_circuit, prepared_variant)
    variant_rows.append({"mixer construction": variant_name, "logical_depth": variant_circuit.depth(),
                         "depth": prepared_variant.metrics["depth"],
                         f"{TWO_QUBIT_GATE.upper()} gates": prepared_variant.metrics["two_qubit_gates"],
                         "swaps": swap_count(prepared_variant.isa_circuit),
                         "est_success_prob": prepared_variant.metrics["estimated_success_probability"]})
print(pd.DataFrame(variant_rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
check("B: no routing SWAPs were inserted for either construction", all(row["swaps"] == 0 for row in variant_rows))

best_variant = min(variant_prepared, key=lambda name: (variant_prepared[name][1].metrics["two_qubit_gates"],
                                                       variant_prepared[name][1].metrics["depth"]))
print(f"Variant used for hardware: {best_variant}")
circuit_b = variant_prepared[best_variant][0]
native_flag_for_b = (best_variant == "XXPlusYYGate")
for layout_name, qubits in chain_layouts.items():
    PREPARED[f"B_{layout_name}"] = prepare_circuit(f"B_{layout_name}", "B", circuit_b, qubits, layout_name)

# Depth study: evaluate both noise-free-optimized parameter sets on the same best physical chain.
QAOA_DEPTH_LABELS = {}
for layers in (1, 2):
    label = f"B_p{layers}_best"
    gammas = qaoa_ideal_results[layers]["gammas"]
    betas = qaoa_ideal_results[layers]["betas"]
    logical = build_xy_qaoa_circuit(problem, gammas, betas, measure=True, native_xy=native_flag_for_b)
    PREPARED[label] = prepare_circuit(label, "B", logical, chain_layouts["best"], "best")
    QAOA_DEPTH_LABELS[layers] = label
    PREPARED[f"cal0_{label}"] = prepare_circuit(f"cal0_{label}", "calibration", calibration_zero, chain_layouts["best"], "best")
    PREPARED[f"cal1_{label}"] = prepare_circuit(f"cal1_{label}", "calibration", calibration_one, chain_layouts["best"], "best")
check("B: p=1 and p=2 high-shot circuits and matched calibrations are native on the same best chain",
      all(PREPARED[label].metrics["native_checked"] and PREPARED[f"cal0_{label}"].metrics["native_checked"]
          and PREPARED[f"cal1_{label}"].metrics["native_checked"] for label in QAOA_DEPTH_LABELS.values()))
metrics_table = pd.DataFrame([
    {"circuit": p.label, "experiment": p.experiment, "layout": p.layout_name,
     "physical_qubits": p.metrics["physical_qubits"], "depth": p.metrics["depth"],
     "two_qubit_gates": p.metrics["two_qubit_gates"], "total_gates": p.metrics["total_gates"],
     "logical_depth": p.metrics["logical_depth"], "logical_2q": p.metrics["logical_two_qubit_gates"],
     "duration_us": p.metrics["estimated_duration_s"] * 1e6,
     "est_success_prob": p.metrics["estimated_success_probability"]}
    for p in PREPARED.values() if p.experiment != "calibration"])
print()
print(metrics_table.to_string(index=False))
VALIDATION_PASSED = all(entry["passed"] for entry in VALIDATION_LOG)

### Noisy Aer simulation of the transpiled circuits
`AerSimulator.from_backend` builds a noise model (gate, readout and relaxation errors) from the backend's calibration. The same transpiled circuits that go to the hardware are simulated, so this is the most direct pre-hardware prediction. We also test, *in simulation*, whether the readout-mitigation procedure (section 11) helps before trusting it on hardware.

In [ ]:
@dataclass
class RunRecord:
    """One measured (or simulated) result; counts are always the raw measurements."""
    label: str
    source: str                      # "ideal_sim", "noisy_sim", "hardware"
    experiment: str
    layout_name: str
    shots: int
    counts: dict
    backend: str = ""
    job_id: str = ""
    job_label: str = ""
    physical_qubits: list = field(default_factory=list)
    suppression: str = "none"
    notes: str = ""


RUNS = {}
noisy_simulator = AerSimulator.from_backend(backend)
SIMULATION_SEED_OFFSETS = {label: index for index, label in enumerate(PREPARED)}


def run_noisy_simulation(label, shots):
    prepared = PREPARED[label]
    seed = RANDOM_SEED + SIMULATION_SEED_OFFSETS[label]
    counts = noisy_simulator.run(prepared.isa_circuit, shots=shots, seed_simulator=seed).result().get_counts()
    RUNS[f"noisy_sim:{label}"] = RunRecord(label, "noisy_sim", prepared.experiment, prepared.layout_name, shots,
                                           dict(counts), backend=f"AerSimulator.from_backend({backend.name})",
                                           physical_qubits=prepared.metrics["physical_qubits"])


for name in single_qubit_layouts:
    run_noisy_simulation(f"A_{name}", SHOTS_STAGE3)
    run_noisy_simulation(f"cal0_A_{name}", SHOTS_CALIBRATION)
    run_noisy_simulation(f"cal1_A_{name}", SHOTS_CALIBRATION)
for name in chain_layouts:
    run_noisy_simulation(f"B_{name}", SHOTS_QAOA)
    run_noisy_simulation(f"cal0_B_{name}", SHOTS_CALIBRATION)
    run_noisy_simulation(f"cal1_B_{name}", SHOTS_CALIBRATION)

# Matched backend-noisy Aer references for the hardware depth study.
for layers, label in QAOA_DEPTH_LABELS.items():
    run_noisy_simulation(label, SHOTS_QAOA_HARDWARE)
    run_noisy_simulation(f"cal0_{label}", SHOTS_QAOA_HARDWARE_CALIBRATION)
    run_noisy_simulation(f"cal1_{label}", SHOTS_QAOA_HARDWARE_CALIBRATION)

# Ideal simulation records (the same logical circuits, no noise).
RUNS["ideal_sim:A"] = RunRecord("A", "ideal_sim", "A", "none", SHOTS_STAGE3, dict(ideal_counts_a), backend="AerSimulator (ideal)")
RUNS["ideal_sim:B"] = RunRecord("B", "ideal_sim", "B", "none", SHOTS_QAOA, dict(ideal_counts_b), backend="AerSimulator (ideal)")
for layers, label in QAOA_DEPTH_LABELS.items():
    RUNS[f"ideal_sim:{label}"] = RunRecord(label, "ideal_sim", "B", "best", SHOTS_QAOA_HARDWARE,
                                            dict(IDEAL_QAOA_DEPTH_COUNTS[label]), backend="AerSimulator (ideal; depth comparison)")

for name in single_qubit_layouts:
    record = RUNS[f"noisy_sim:A_{name}"]
    print(f"Noisy sim A on '{name}' qubits {record.physical_qubits}: P(K>=3) = {tail_from_counts(record.counts):.4f} "
          f"(exact {EXACT_TAIL:.4f})")
for name in chain_layouts:
    vector = counts_to_probability_vector(RUNS[f"noisy_sim:B_{name}"].counts, problem.size)
    metrics = selection_metrics(vector)
    print(f"Noisy sim B on '{name}' path: feasibility {metrics['feasibility']:.3f}, "
          f"P(optimum) {metrics['optimal_probability']:.3f}, approx ratio {metrics['approximation_ratio']:.3f}")

## 11. Actual IBM hardware execution
### Readout-error mitigation (implemented locally)
Measuring $\lvert 0\ldots0\rangle$ and $\lvert 1\ldots1\rangle$ gives each qubit's assignment errors $\epsilon_{0\to1}$ and $\epsilon_{1\to0}$. Assuming independent readout errors, the confusion matrix is a tensor product of $2\times 2$ matrices $M_i=\begin{pmatrix}1-\epsilon_{0\to1}&\epsilon_{1\to0}\\ \epsilon_{0\to1}&1-\epsilon_{1\to0}\end{pmatrix}$ and the mitigated distribution is $\bigotimes_i M_i^{-1}$ applied to the measured one (negative entries clipped, then renormalised). It corrects only *independent* readout error, not gate errors or correlated readout error.

### Budget and job-safety model (next cells)
* The balance is **never assumed**. You enter `REMAINING_QPU_SECONDS` from the IBM dashboard; if the account API also reports a balance the smaller value is used. Unknown balance means **simulation only**.
* A reserve of `max(QPU_MIN_RESERVE_SECONDS, QPU_RESERVE_FRACTION × balance)` is never planned for. Planned cost = estimate × learned scale (never below 1) × `BUDGET_UNCERTAINTY_FACTOR`, and a single job may not exceed `MAX_PLANNED_SECONDS_PER_JOB`.
* Every submitted job is written to `raw/job_registry.json` **before** waiting. A job that is queued, running, timed out or finished-without-reported-usage is charged its planned cost until its actual usage is reconciled.
* A Python timeout does **not** stop the IBM job. The job ID is kept, re-running the notebook resumes it, and **no new job is submitted while any job is unfinished or unreconciled**.
* Jobs after the first need job 1 collected and reconciled. Priority: job 1 (Stage 3), job 2 (Stage 2 synthetic), then comparisons.

In [ ]:
def estimate_assignment_errors(zero_vector, one_vector):
    """Per-qubit P(read 1 | prepared 0) and P(read 0 | prepared 1) from outcome-probability vectors."""
    indices = np.arange(2 ** NUM_CUSTOMERS)
    qubit_is_one = lambda qubit: ((indices >> qubit) & 1).astype(bool)
    flip_0_to_1 = np.array([zero_vector[qubit_is_one(q)].sum() for q in range(NUM_CUSTOMERS)])
    flip_1_to_0 = np.array([one_vector[~qubit_is_one(q)].sum() for q in range(NUM_CUSTOMERS)])
    return flip_0_to_1, flip_1_to_0


def apply_per_qubit_matrices(vector, matrices):
    """Apply a 2x2 matrix to each qubit of a 2^n vector (qubit i = bit i of the index)."""
    n = NUM_CUSTOMERS
    tensor = vector.reshape((2,) * n)                  # axis 0 = most significant bit = qubit n-1
    for qubit in range(n):
        axis = n - 1 - qubit
        tensor = np.moveaxis(np.tensordot(matrices[qubit], tensor, axes=([1], [axis])), 0, axis)
    return tensor.reshape(-1)


def confusion_matrices(flip_0_to_1, flip_1_to_0):
    return [np.array([[1 - flip_0_to_1[q], flip_1_to_0[q]],
                      [flip_0_to_1[q], 1 - flip_1_to_0[q]]]) for q in range(NUM_CUSTOMERS)]


def mitigate_outcome_vector(measured_vector, flip_0_to_1, flip_1_to_0):
    """Inverse tensor-product confusion, then clip negatives and renormalise."""
    inverses = [np.linalg.inv(m) for m in confusion_matrices(flip_0_to_1, flip_1_to_0)]
    mitigated = np.clip(apply_per_qubit_matrices(measured_vector, inverses), 0.0, None)
    return mitigated / mitigated.sum()


# Unit test with a known synthetic confusion: mitigation must recover the exact distribution.
test_flip_01 = np.array([0.01, 0.02, 0.03, 0.015, 0.025, 0.05])
test_flip_10 = np.array([0.03, 0.04, 0.02, 0.06, 0.05, 0.08])
blurred = apply_per_qubit_matrices(exact_outcomes, confusion_matrices(test_flip_01, test_flip_10))
check("Readout mitigation exactly inverts a known confusion (qubit ordering verified)",
      np.allclose(mitigate_outcome_vector(blurred, test_flip_01, test_flip_10), exact_outcomes, atol=1e-10))
perfect_zero = np.zeros(2 ** NUM_CUSTOMERS); perfect_zero[0] = 1.0
perfect_one = np.zeros(2 ** NUM_CUSTOMERS); perfect_one[-1] = 1.0
estimated_01, estimated_10 = estimate_assignment_errors(perfect_zero, perfect_one)
check("Calibration estimator gives zero error for perfect all-0 / all-1 data",
      estimated_01.sum() == 0 and estimated_10.sum() == 0)


def mitigate_with_calibration(measured_vector, zero_vector, one_vector):
    flip_01, flip_10 = estimate_assignment_errors(zero_vector, one_vector)
    return mitigate_outcome_vector(measured_vector, flip_01, flip_10)


def tail_from_vector(vector):
    return float(vector[POPULATION_COUNT >= SUCCESS_THRESHOLD].sum())


# Does mitigation help in *simulation*? (informational; hardware may behave differently)
for name in single_qubit_layouts:
    raw_tail = tail_from_counts(RUNS[f"noisy_sim:A_{name}"].counts)
    fixed_tail = tail_from_vector(mitigate_with_calibration(
        counts_to_probability_vector(RUNS[f"noisy_sim:A_{name}"].counts, NUM_CUSTOMERS),
        counts_to_probability_vector(RUNS[f"noisy_sim:cal0_A_{name}"].counts, NUM_CUSTOMERS),
        counts_to_probability_vector(RUNS[f"noisy_sim:cal1_A_{name}"].counts, NUM_CUSTOMERS)))
    print(f"Noisy sim, layout '{name}': |error| raw {abs(raw_tail - EXACT_TAIL):.4f} -> mitigated {abs(fixed_tail - EXACT_TAIL):.4f}")
VALIDATION_PASSED = all(entry["passed"] for entry in VALIDATION_LOG)

In [ ]:
FINAL_STATES = {"DONE", "CANCELLED", "ERROR"}
STATE_ALIASES = {"COMPLETED": "DONE", "FAILED": "ERROR", "CANCELED": "CANCELLED"}
REGISTRY_PATH = RAW_DIR / "job_registry.json"
SESSION_ID = START_TIME_UTC
FIRST_JOB_LABEL = "job1_stage3_best_layout"
NEW_JOBS_SUBMITTED = 0
ERRORS = []
RUNS_RESTORED = 0


def load_registry():
    if not REGISTRY_PATH.exists():
        return {}
    try:
        with open(REGISTRY_PATH) as handle:
            return json.load(handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(f"{REGISTRY_PATH} is corrupt ({error}). Check the IBM Quantum dashboard for submitted jobs, "
                           "then repair or delete the file by hand. Nothing will be submitted until then.") from None


JOB_REGISTRY = load_registry()


def persist_registry():
    save_json(JOB_REGISTRY, REGISTRY_PATH)


def job_ids():
    return {label: record["job_id"] for label, record in JOB_REGISTRY.items()}


# --------------------------------------------------------------------------- job plan
def estimate_job_seconds(entries):
    """Rough QPU seconds for a list of (prepared-circuit label, shots); deliberately generous."""
    repetition_delay = getattr(backend, "default_rep_delay", None) or 250e-6
    total = FIXED_JOB_OVERHEAD_SECONDS
    for label, shots in entries:
        duration = PREPARED[label].metrics["estimated_duration_s"]
        duration = FALLBACK_CIRCUIT_DURATION_SECONDS if math.isnan(duration) else duration
        total += CIRCUIT_OVERHEAD_SECONDS + shots * (repetition_delay + duration)
    return total


JOB_PLAN = {
    "job1_stage3_best_layout": {
        "priority": 1, "experiment": "A (real Stage 3 data)", "suppression": "none",
        "entries": [("A_best", SHOTS_STAGE3), ("cal0_A_best", SHOTS_CALIBRATION), ("cal1_A_best", SHOTS_CALIBRATION)]},
    "job2_stage2_synthetic_best_chain": {
        "priority": 2, "experiment": "B (SYNTHETIC instance)", "suppression": "none",
        "entries": [(QAOA_DEPTH_LABELS[1], SHOTS_QAOA_HARDWARE), (f"cal0_{QAOA_DEPTH_LABELS[1]}", SHOTS_QAOA_HARDWARE_CALIBRATION), (f"cal1_{QAOA_DEPTH_LABELS[1]}", SHOTS_QAOA_HARDWARE_CALIBRATION), (QAOA_DEPTH_LABELS[2], SHOTS_QAOA_HARDWARE), (f"cal0_{QAOA_DEPTH_LABELS[2]}", SHOTS_QAOA_HARDWARE_CALIBRATION), (f"cal1_{QAOA_DEPTH_LABELS[2]}", SHOTS_QAOA_HARDWARE_CALIBRATION)]},
    "job3_stage3_other_layouts": {
        "priority": 3, "experiment": "C: A on middle and poor qubits", "suppression": "none",
        "entries": [("A_middle", SHOTS_STAGE3), ("cal0_A_middle", SHOTS_CALIBRATION), ("cal1_A_middle", SHOTS_CALIBRATION),
                    ("A_poor", SHOTS_STAGE3), ("cal0_A_poor", SHOTS_CALIBRATION), ("cal1_A_poor", SHOTS_CALIBRATION)]},
    "job4_stage2_synthetic_typical_chain": {
        "priority": 3, "experiment": "C: B (SYNTHETIC) on a typical chain", "suppression": "none",
        "entries": [("B_typical", SHOTS_QAOA), ("cal0_B_typical", SHOTS_CALIBRATION), ("cal1_B_typical", SHOTS_CALIBRATION)]},
    "job5_suppression": {
        "priority": 4, "experiment": "D: DD + twirling on B (SYNTHETIC) and A", "suppression": "dd_twirl",
        "entries": [("B_best", SHOTS_QAOA), ("B_typical", SHOTS_QAOA), ("A_best", SHOTS_STAGE3)]},
}
for repeat_index in range(1, REPEATABILITY_RUNS + 1):
    JOB_PLAN[f"job6_repeat_{repeat_index}"] = {
        "priority": 4, "experiment": "E: repeat of A on the best qubits", "suppression": "none",
        "entries": [("A_best", SHOTS_STAGE3)]}


# --------------------------------------------------------------------------- budget
class QpuBudget:
    """Conservative accounting of QPU seconds, driven by the persistent job registry."""

    def __init__(self, manual_remaining, account_remaining):
        known = [float(v) for v in (manual_remaining, account_remaining) if v is not None]
        if any(v < 0 for v in known):
            raise ValueError("Remaining QPU seconds cannot be negative.")
        self.manual_remaining = manual_remaining
        self.account_remaining = account_remaining
        self.start_remaining = min(known) if known else None

    @property
    def known(self):
        return self.start_remaining is not None

    @property
    def allowance(self):
        return self.start_remaining if self.start_remaining is not None else 0.0

    @property
    def reserve(self):
        return max(QPU_MIN_RESERVE_SECONDS, QPU_RESERVE_FRACTION * self.allowance)

    def counted_seconds(self, record):
        """Actual usage if reconciled, otherwise the conservative planned cost."""
        if record.get("usage_reconciled"):
            return float(record["actual_seconds"])
        return float(record.get("planned_cost_seconds") or 0.0)

    def counts_against_balance(self, record):
        if not BALANCE_INCLUDES_PREVIOUS_JOBS:
            return True
        return record.get("session") == SESSION_ID or not record.get("usage_reconciled")

    @property
    def used_seconds(self):
        return sum(self.counted_seconds(r) for r in JOB_REGISTRY.values() if self.counts_against_balance(r))

    @property
    def spendable(self):
        return self.allowance - self.reserve - self.used_seconds

    @property
    def cost_scale(self):
        ratios = [r["actual_seconds"] / r["estimated_seconds"] for r in JOB_REGISTRY.values()
                  if r.get("usage_reconciled") and r.get("estimated_seconds")]
        return max([1.0] + ratios)

    def planned_cost(self, estimated_seconds):
        return estimated_seconds * self.cost_scale * BUDGET_UNCERTAINTY_FACTOR

    @property
    def ledger(self):
        rows, cumulative = [], 0.0
        for label, record in sorted(JOB_REGISTRY.items(), key=lambda item: item[1].get("submitted_utc", "")):
            counted = self.counted_seconds(record)
            cumulative += counted
            rows.append({"job_label": label, "job_id": record["job_id"], "status": record.get("status"),
                         "estimated_seconds": record.get("estimated_seconds"), "planned_cost_seconds": record.get("planned_cost_seconds"),
                         "actual_seconds": record.get("actual_seconds"), "usage_reconciled": bool(record.get("usage_reconciled")),
                         "counted_seconds": counted, "cumulative_seconds": cumulative})
        return rows

    def summary(self):
        return {"manual_remaining_seconds": self.manual_remaining, "account_remaining_seconds": self.account_remaining,
                "balance_used_for_planning": self.start_remaining, "balance_known": self.known, "reserve_seconds": self.reserve,
                "counted_used_seconds": self.used_seconds, "spendable_seconds": self.spendable, "cost_scale": self.cost_scale,
                "uncertainty_factor": BUDGET_UNCERTAINTY_FACTOR, "balance_includes_previous_jobs": BALANCE_INCLUDES_PREVIOUS_JOBS}


budget = QpuBudget(REMAINING_QPU_SECONDS, account_remaining_seconds)


# --------------------------------------------------------------------------- gates
def hardware_gate_failures():
    """Reasons hardware mode is off (empty list = armed). Never raises."""
    reasons = []
    if not ENABLE_HARDWARE:
        reasons.append("ENABLE_HARDWARE is False")
    if HARDWARE_CONFIRMATION_TEXT != REQUIRED_CONFIRMATION_TEXT:
        reasons.append("HARDWARE_CONFIRMATION_TEXT does not match REQUIRED_CONFIRMATION_TEXT")
    if service is None or backend_is_fake:
        reasons.append("no IBM login / only the local fake backend is available")
    if not HARDWARE_PREFLIGHT_OK:
        reasons.append("read-only IBM hardware preflight did not confirm an accessible operational QPU")
    if not VALIDATION_PASSED:
        reasons.append("a validation check failed")
    if not PROBABILITIES_VERIFIED:
        reasons.append("Stage 3 probabilities are not verified against the reference")
    if not budget.known:
        reasons.append("QPU balance unknown: authenticated service usage did not provide remaining seconds; set REMAINING_QPU_SECONDS from the current IBM dashboard")
    return reasons


def submission_blockers(job_label):
    """Every reason this particular job may not be submitted right now (empty list = allowed)."""
    reasons = hardware_gate_failures()
    plan = JOB_PLAN.get(job_label)
    if plan is None:
        return reasons + [f"'{job_label}' is not in JOB_PLAN"]
    if job_label not in CONFIRMED_JOBS:
        reasons.append(f"'{job_label}' is not in CONFIRMED_JOBS")
    if NEW_JOBS_SUBMITTED >= MAX_NEW_JOBS_PER_RUN:
        reasons.append(f"already submitted {NEW_JOBS_SUBMITTED} new job(s) this run (MAX_NEW_JOBS_PER_RUN={MAX_NEW_JOBS_PER_RUN})")
    if job_label in JOB_REGISTRY:
        reasons.append(f"'{job_label}' already exists in the registry (status {JOB_REGISTRY[job_label].get('status')}); "
                       "it is resumed, never resubmitted. For a deliberate retry check the IBM dashboard, then remove its registry entry.")
    for other_label, record in JOB_REGISTRY.items():
        if other_label == job_label:
            continue
        if record.get("status") not in FINAL_STATES:
            reasons.append(f"job '{other_label}' ({record['job_id']}) is still {record.get('status')}; its cost is unknown")
        elif not record.get("usage_reconciled"):
            reasons.append(f"job '{other_label}' ({record['job_id']}) finished but its QPU usage is not reconciled "
                           "(enter it in MANUAL_USAGE_OVERRIDES from the dashboard)")
        if record.get("backend") and record["backend"] != backend.name:
            reasons.append(f"job '{other_label}' ran on {record['backend']} but the selected backend is {backend.name}")
    if job_label != FIRST_JOB_LABEL:
        first = JOB_REGISTRY.get(FIRST_JOB_LABEL, {})
        if not (first.get("results_saved") and first.get("usage_reconciled")):
            reasons.append(f"'{FIRST_JOB_LABEL}' must be collected and reconciled before any other job")
    if budget.known:
        planned = budget.planned_cost(estimate_job_seconds(plan["entries"]))
        if planned > MAX_PLANNED_SECONDS_PER_JOB:
            reasons.append(f"planned cost {planned:.1f}s exceeds MAX_PLANNED_SECONDS_PER_JOB={MAX_PLANNED_SECONDS_PER_JOB}")
        if planned > budget.spendable:
            reasons.append(f"planned cost {planned:.1f}s exceeds spendable budget {budget.spendable:.1f}s")
    return reasons


def plan_table():
    rows = []
    for label, plan in JOB_PLAN.items():
        estimate = estimate_job_seconds(plan["entries"])
        record = JOB_REGISTRY.get(label, {})
        blockers = submission_blockers(label)
        rows.append({"job": label, "priority": plan["priority"], "experiment": plan["experiment"],
                     "circuits": len(plan["entries"]), "shots": sum(s for _, s in plan["entries"]),
                     "estimate_s": round(estimate, 1), "planned_cost_s": round(budget.planned_cost(estimate), 1),
                     "registry_status": record.get("status", "-"),
                     "can_submit_now": not blockers})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- job handling
def job_state(job):
    state = str(job.status()).split(".")[-1].strip().upper()
    return STATE_ALIASES.get(state, state)


def read_job_usage_seconds(job, retries=4):
    """Actual QPU seconds reported by IBM for a finished job (None if unavailable)."""
    override = MANUAL_USAGE_OVERRIDES.get(job.job_id())
    if override is not None:
        return float(override)
    for _ in range(retries):
        try:
            usage = job.usage()
            if usage is not None:
                return float(usage)
        except Exception:
            pass
        time.sleep(3)
    return None


def reconcile_usage(job, record):
    actual = read_job_usage_seconds(job)
    if actual is None:
        record["usage_reconciled"] = False
        print(f"  usage of {record['job_id']} not reported yet: charged the planned {record['planned_cost_seconds']:.1f}s and "
              "blocking further submissions. Re-run later or set MANUAL_USAGE_OVERRIDES from the dashboard.")
    else:
        record["actual_seconds"] = actual
        record["usage_reconciled"] = True
        print(f"  actual QPU usage: {actual:.2f}s (estimated {record['estimated_seconds']:.1f}s)")
    persist_registry()


def raw_run_path(circuit_label, job_label):
    return RAW_DIR / ("counts__" + f"hardware:{circuit_label}:{job_label}".replace(":", "__") + ".json")


def collect_results(job, record):
    """Retrieve and store the raw counts of a DONE job. Returns True on success."""
    plan = JOB_PLAN[record["job_label"]]
    result = job.result()
    if len(result) != len(plan["entries"]):
        raise RuntimeError(f"job returned {len(result)} results but the plan has {len(plan['entries'])} circuits")
    for (circuit_label, shots), pub_result in zip(plan["entries"], result):
        counts = dict(pub_result.data.meas.get_counts())
        measured_shots = sum(counts.values())
        prepared = PREPARED[circuit_label]
        run = RunRecord(circuit_label, "hardware", prepared.experiment, prepared.layout_name, measured_shots, counts,
                        backend=record.get("backend", backend.name), job_id=record["job_id"], job_label=record["job_label"],
                        physical_qubits=record.get("physical_qubits_by_label", {}).get(
                            circuit_label, prepared.metrics["physical_qubits"]),
                        suppression=plan["suppression"],
                        notes="" if measured_shots == shots else f"requested {shots} shots, measured {measured_shots}")
        RUNS[f"hardware:{circuit_label}:{record['job_label']}"] = run
        save_json(asdict(run), raw_run_path(circuit_label, record["job_label"]))
    record["results_saved"] = True
    persist_registry()
    return True


def wait_and_collect(job, record, max_wait_minutes):
    """Poll one job, collect it when DONE, reconcile usage. Never submits anything."""
    deadline = time.time() + 60 * max_wait_minutes
    last_state, failures = None, 0
    while True:
        try:
            state = job_state(job)
            failures = 0
        except Exception as error:
            failures += 1
            print(f"  status check failed ({type(error).__name__}), attempt {failures}")
            if failures >= 5:
                state = record.get("status", "UNKNOWN")
                print("  giving up for now; the job may still be running on IBM.")
                return state
            time.sleep(POLL_SECONDS)
            continue
        record["status"] = state
        persist_registry()
        if state != last_state:
            print(f"  [{record['job_label']}] job {record['job_id']} status: {state}")
            last_state = state
        if state in FINAL_STATES:
            break
        if time.time() >= deadline:
            print(f"  Stopped waiting ({max_wait_minutes} min). Job {record['job_id']} is NOT cancelled and may still run or finish. "
                  "Re-run this cell later to collect it. No other job will be submitted meanwhile.")
            return state
        time.sleep(POLL_SECONDS)
    if state == "DONE" and not record.get("results_saved"):
        try:
            collect_results(job, record)
        except Exception as error:
            ERRORS.append({"job_label": record["job_label"], "job_id": record["job_id"],
                           "error_type": type(error).__name__, "error": str(error)})
            save_json(ERRORS, RAW_DIR / "errors.json")
            print(f"  result retrieval failed: {type(error).__name__}: {error}")
    elif state != "DONE":
        try:
            message = job.error_message()
        except Exception:
            message = None
        ERRORS.append({"job_label": record["job_label"], "job_id": record["job_id"], "status": state, "message": message})
        save_json(ERRORS, RAW_DIR / "errors.json")
        print(f"  job ended as {state}. {message or ''}")
    if not record.get("usage_reconciled"):
        reconcile_usage(job, record)
    return state


def restore_saved_runs():
    """Reload raw counts of already-collected hardware jobs (after a restart with persistent storage)."""
    restored = 0
    for job_label, record in JOB_REGISTRY.items():
        if not record.get("results_saved") or job_label not in JOB_PLAN:
            continue
        paths = [raw_run_path(label, job_label) for label, _ in JOB_PLAN[job_label]["entries"]]
        if not all(p.exists() for p in paths):
            record["results_saved"] = False       # files lost: fetch again from IBM
            continue
        for path in paths:
            with open(path) as handle:
                data = json.load(handle)
            RUNS[f"hardware:{data['label']}:{job_label}"] = RunRecord(**data)
            restored += 1
    persist_registry()
    return restored


def sync_registry():
    """Read-only refresh of every registered job (status, results, usage). Safe to run any time."""
    if not JOB_REGISTRY:
        print("Job registry is empty: no hardware job has been submitted from this folder.")
        return
    for job_label, record in list(JOB_REGISTRY.items()):
        done = record.get("results_saved") and record.get("usage_reconciled")
        if done or job_label not in JOB_PLAN:
            print(f"  {job_label}: {record.get('status')} {'(complete)' if done else '(not in JOB_PLAN, skipped)'}")
            continue
        if service is None:
            print(f"  {job_label}: {record.get('status')} (no IBM login, cannot refresh)")
            continue
        try:
            wait_and_collect(service.job(record["job_id"]), record, max_wait_minutes=0)
        except Exception as error:
            print(f"  {job_label}: refresh failed ({type(error).__name__}); job {record['job_id']} is unchanged on IBM")


def confirm_submission(job_label, planned_cost):
    if not REQUIRE_INTERACTIVE_CONFIRMATION:
        return True
    try:
        answer = input(f"Submit '{job_label}' to {backend.name}? Planned QPU cost up to {planned_cost:.1f}s. Type YES to submit: ")
    except (EOFError, KeyboardInterrupt):
        print("  no confirmation received: nothing submitted.")
        return False
    return answer.strip() == "YES"


def submit_job(job_label):
    global NEW_JOBS_SUBMITTED
    plan = JOB_PLAN[job_label]
    estimate = estimate_job_seconds(plan["entries"])
    planned = budget.planned_cost(estimate)
    print(f"[{job_label}] {plan['experiment']}: estimate {estimate:.1f}s, planned cost {planned:.1f}s, "
          f"spendable {budget.spendable:.1f}s of {budget.allowance:.0f}s (reserve {budget.reserve:.0f}s)")
    if not confirm_submission(job_label, planned):
        return None
    NEW_JOBS_SUBMITTED += 1                       # counted before the call, so a failure cannot cause a retry loop
    sampler = SamplerV2(mode=backend)
    if plan["suppression"] == "dd_twirl":
        sampler.options.dynamical_decoupling.enable = True
        sampler.options.dynamical_decoupling.sequence_type = "XpXm"
        sampler.options.twirling.enable_gates = True
        sampler.options.twirling.enable_measure = True
    job = sampler.run([(PREPARED[label].isa_circuit, None, shots) for label, shots in plan["entries"]])
    JOB_REGISTRY[job_label] = {
        "job_label": job_label, "job_id": job.job_id(), "backend": backend.name, "experiment": plan["experiment"],
        "entries": [list(e) for e in plan["entries"]], "suppression": plan["suppression"],
        "submitted_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "session": SESSION_ID,
        "estimated_seconds": estimate, "planned_cost_seconds": planned, "status": "SUBMITTED",
        "usage_reconciled": False, "actual_seconds": None, "results_saved": False}
    persist_registry()                            # job id is on disk before we wait
    print(f"  submitted job {job.job_id()}")
    return job


def run_job(job_label):
    """Collect, resume or (only if every gate allows) submit one planned job."""
    if job_label not in JOB_PLAN:
        raise KeyError(f"unknown job label {job_label}")
    record = JOB_REGISTRY.get(job_label)
    if record is None and job_label in RESUME_JOB_IDS:
        estimate = estimate_job_seconds(JOB_PLAN[job_label]["entries"])
        resume_metadata = RESUME_JOB_METADATA.get(job_label, {})
        record = JOB_REGISTRY[job_label] = {
            "job_label": job_label, "job_id": RESUME_JOB_IDS[job_label],
            "backend": resume_metadata.get("backend", backend.name),
            "physical_qubits_by_label": resume_metadata.get("physical_qubits_by_label", {}),
            "experiment": JOB_PLAN[job_label]["experiment"], "entries": [list(e) for e in JOB_PLAN[job_label]["entries"]],
            "suppression": JOB_PLAN[job_label]["suppression"], "submitted_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "session": SESSION_ID, "estimated_seconds": estimate, "planned_cost_seconds": budget.planned_cost(estimate),
            "status": "UNKNOWN", "usage_reconciled": False, "actual_seconds": None, "results_saved": False,
            "note": "resumed from RESUME_JOB_IDS; counted against the budget like any other job"}
        persist_registry()
    if record is not None:
        if record.get("results_saved") and record.get("usage_reconciled"):
            print(f"[{job_label}] already complete (job {record['job_id']}).")
            return record["status"]
        if service is None:
            print(f"[{job_label}] registered as {record.get('status')} (job {record['job_id']}) but there is no IBM login to refresh it.")
            return record.get("status")
        print(f"[{job_label}] resuming existing job {record['job_id']} (no new submission).")
        return wait_and_collect(service.job(record["job_id"]), record, MAX_QUEUE_WAIT_MINUTES)
    blockers = submission_blockers(job_label)
    if blockers:
        print(f"[{job_label}] NOT submitted:")
        for reason in blockers:
            print(f"   - {reason}")
        return "blocked"
    try:
        job = submit_job(job_label)
    except Exception as error:
        ERRORS.append({"job_label": job_label, "error_type": type(error).__name__, "error": str(error)})
        save_json(ERRORS, RAW_DIR / "errors.json")
        print(f"[{job_label}] submission failed: {type(error).__name__}: {error}. Check the IBM dashboard before trying again.")
        return "submit_failed"
    if job is None:
        return "declined"
    return wait_and_collect(job, JOB_REGISTRY[job_label], MAX_QUEUE_WAIT_MINUTES)

In [ ]:
# Focused loader and hardware-gate regression checks (no job submission).
def test_stage3_customer_id_resolution():
    # Test-only identity sentinels are not probabilities and never enter an experiment.
    sentinel_by_id = {customer_id: 100 + index for index, customer_id in enumerate(EXPECTED_CUSTOMER_IDS)}
    shuffled_ids = [EXPECTED_CUSTOMER_IDS[index] for index in (3, 0, 5, 1, 4, 2)]
    shuffled_values = [sentinel_by_id[customer_id] for customer_id in shuffled_ids]
    resolved_ids, resolved_values, _ = resolve_candidate(shuffled_ids, shuffled_values)
    assert resolved_ids == EXPECTED_CUSTOMER_IDS
    assert dict(zip(resolved_ids, resolved_values)) == sentinel_by_id

    # A full customer table may contain unrelated rows; selected values still map by ID.
    wide_ids = ["C9001", *shuffled_ids, "C9002"]
    wide_values = [999, *shuffled_values, 998]
    wide_resolved = resolve_candidate(wide_ids, wide_values)
    assert wide_resolved[0] == EXPECTED_CUSTOMER_IDS
    assert dict(zip(wide_resolved[0], wide_resolved[1])) == sentinel_by_id

    # The real loaded values also return to the exact canonical order when IDs are present.
    if PROBABILITIES_VERIFIED:
        actual_by_id = dict(zip(stage3_customer_ids, stage3_probabilities.tolist()))
        actual_resolved = resolve_candidate(shuffled_ids, [actual_by_id[cid] for cid in shuffled_ids])
        assert actual_resolved[0] == EXPECTED_CUSTOMER_IDS
        assert actual_resolved[1] == [actual_by_id[cid] for cid in EXPECTED_CUSTOMER_IDS]

    assert resolve_candidate(EXPECTED_CUSTOMER_IDS[:-1], stage3_probabilities.tolist()) is None
    incorrect_ids = EXPECTED_CUSTOMER_IDS[:-1] + ["C9999"]
    assert resolve_candidate(incorrect_ids, stage3_probabilities.tolist()) is None
    duplicate_ids = EXPECTED_CUSTOMER_IDS[:-1] + [EXPECTED_CUSTOMER_IDS[-2]]
    assert resolve_candidate(duplicate_ids, stage3_probabilities.tolist()) is None

    idless = resolve_candidate(None, stage3_probabilities.tolist())
    assert idless is not None and idless[0] is None
    assert not customer_ids_verified(idless[0])
    assert not candidate_problems(idless[0], idless[1])  # exact values may still be simulated


def test_unverified_customer_mapping_blocks_hardware():
    global PROBABILITIES_VERIFIED
    previous = PROBABILITIES_VERIFIED
    try:
        PROBABILITIES_VERIFIED = False
        assert any("Stage 3 probabilities are not verified" in reason
                   for reason in hardware_gate_failures())
    finally:
        PROBABILITIES_VERIFIED = previous



def test_qpu_budget_uses_authenticated_balance_without_manual_override():
    # Authenticated IBM usage seconds are sufficient; manual entry is only a fallback.
    api_budget = QpuBudget(None, 461.4)
    assert api_budget.known
    assert api_budget.allowance == 461.4
    assert api_budget.spendable > 0

    # The smaller figure remains authoritative when both sources are available.
    capped_budget = QpuBudget(300.0, 461.4)
    assert capped_budget.known
    assert capped_budget.allowance == 300.0

    # Unknown balance must remain blocked.
    unknown_budget = QpuBudget(None, None)
    assert not unknown_budget.known
    assert unknown_budget.spendable <= 0


def test_hardware_connection_preflight_is_read_only():
    class MockBackend:
        name = "mock_qpu"

    class MockService:
        def __init__(self):
            self.backend_calls = 0
        def active_instance(self):
            return "mock-instance"
        def backends(self, **kwargs):
            self.backend_calls += 1
            assert kwargs == {"simulator": False, "operational": True, "min_num_qubits": MIN_BACKEND_QUBITS}
            return [MockBackend()]

    mock = MockService()
    ok, instance, backends = hardware_connection_preflight(mock)
    assert ok and instance == "mock-instance" and [b.name for b in backends] == ["mock_qpu"]
    assert mock.backend_calls == 1  # metadata lookup only; mock has no submit/run method
    failed, _, empty = hardware_connection_preflight(None)
    assert not failed and empty == []


def test_qaoa_depth_hardware_job_has_equal_shots_and_matched_calibrations():
    plan = JOB_PLAN["job2_stage2_synthetic_best_chain"]["entries"]
    assert len(plan) == 6
    assert plan[0] == ("B_p1_best", SHOTS_QAOA_HARDWARE)
    assert plan[3] == ("B_p2_best", SHOTS_QAOA_HARDWARE)
    assert plan[1:3] == [(f"cal0_B_p1_best", SHOTS_QAOA_HARDWARE_CALIBRATION),
                         (f"cal1_B_p1_best", SHOTS_QAOA_HARDWARE_CALIBRATION)]
    assert plan[4:6] == [(f"cal0_B_p2_best", SHOTS_QAOA_HARDWARE_CALIBRATION),
                         (f"cal1_B_p2_best", SHOTS_QAOA_HARDWARE_CALIBRATION)]
    assert len(qaoa_ideal_results[1]["gammas"]) == 1
    assert len(qaoa_ideal_results[2]["gammas"]) == 2
    assert all(PREPARED[label].metrics["native_checked"] for label in QAOA_DEPTH_LABELS.values())

test_stage3_customer_id_resolution()
test_unverified_customer_mapping_blocks_hardware()
test_qpu_budget_uses_authenticated_balance_without_manual_override()
test_hardware_connection_preflight_is_read_only()
test_qaoa_depth_hardware_job_has_equal_shots_and_matched_calibrations()
print("PASS: mapping, preflight, budget, and p=1/p=2 QAOA job checks")


### Hardware gate status and planned experiments
Shows whether hardware mode is armed (it is not, by default), the budget numbers, and the plan with its conservative cost estimates. Nothing is submitted by this cell.

In [ ]:
restored_count = restore_saved_runs()
HARDWARE_ENABLED = not hardware_gate_failures()
failures = hardware_gate_failures()
print("=" * 78)
if failures:
    print("MODE: SIMULATION ONLY. No IBM job will be submitted. Reasons:")
    for reason in failures:
        print(f"  - {reason}")
else:
    print(f"MODE: HARDWARE ARMED on {backend.name}. Jobs still need CONFIRMED_JOBS, a typed YES, and budget.")
print(f"Balance for planning: {budget.start_remaining if budget.known else 'UNKNOWN'} s | reserve {budget.reserve:.0f} s | "
      f"counted so far {budget.used_seconds:.1f} s | spendable {budget.spendable if budget.known else 0.0:.1f} s")
print(f"Restored hardware records from disk: {restored_count} | registry jobs: {len(JOB_REGISTRY)}")
print("=" * 78)
print(plan_table().to_string(index=False))

### Synchronise with IBM (read-only)
Refreshes every job in the registry (status, results, actual usage). Needed after a Colab disconnect. It never submits anything and does not wait for queued jobs.

In [ ]:
sync_registry()
HARDWARE_ENABLED = not hardware_gate_failures()

### Job 1 — Experiment A on the best qubits (priority 1, real Stage 3 data)
The Stage 3 circuit plus its two readout-calibration circuits, all on the same six physical qubits. No two-qubit gates. After it finishes the **actual** QPU usage is read back and drives every later decision. To run it: set `ENABLE_HARDWARE`, `HARDWARE_CONFIRMATION_TEXT`, `REMAINING_QPU_SECONDS` and `CONFIRMED_JOBS = {"job1_stage3_best_layout"}` in section 2, re-run, and type YES when asked.

In [ ]:
job1_status = run_job("job1_stage3_best_layout")

### Job 2 - QAOA depth comparison on the best chain (priority 2, **SYNTHETIC** instance)

Runs fixed, noise-free-optimized exact-cardinality XY-QAOA parameters at p=1 and p=2 on the same six-qubit physical chain. Each depth receives 30,000 QPU shots plus matched 8,000-shot all-zero/all-one readout calibrations. The notebook records ideal statevector, equal-shot ideal Aer, backend-noisy Aer, raw hardware, and readout-mitigated hardware results side by side. This evaluates fixed parameters; it is not hardware-in-the-loop parameter optimization. The optimization instance is synthetic, not Atomiq customer data.


In [ ]:
job2_status = run_job("job2_stage2_synthetic_best_chain")

### Job 3 — Experiment C for Stage 3: middle and poor qubit groups (priority 3, optional)
Same circuit and shots as job 1, only the physical qubits differ.

In [ ]:
job3_status = run_job("job3_stage3_other_layouts")

### Job 4 — Experiment C for the synthetic QAOA: typical chain (priority 3, optional)

In [ ]:
job4_status = run_job("job4_stage2_synthetic_typical_chain")

### Job 5 — Experiment D: dynamical decoupling and Pauli twirling (priority 4, optional)
Identical circuits and layouts, with XpXm dynamical decoupling plus gate and measurement twirling switched on in the Sampler.

In [ ]:
job5_status = run_job("job5_suppression")

### Job 6 — Experiment E: repeatability of Experiment A (priority 4, optional)
Repeats the baseline Stage 3 measurement in a separate job. Disagreement beyond shot noise indicates drift. Only meaningful if job 1 and the repeat used the same backend and layout.

In [ ]:
for repeat_index in range(1, REPEATABILITY_RUNS + 1):
    run_job(f"job6_repeat_{repeat_index}")

### Hardware summary

In [ ]:
hardware_runs = {key: run for key, run in RUNS.items() if run.source == "hardware"}
print(f"Hardware records collected: {len(hardware_runs)} | new jobs submitted this run: {NEW_JOBS_SUBMITTED}")
print(f"Job IDs: {job_ids()}")
print(f"Counted QPU usage: {budget.used_seconds:.1f} s | balance used for planning: {budget.start_remaining} s")
if not hardware_runs:
    print("No hardware result exists yet. Every table and figure below shows simulation only and is labelled as such.")
if ERRORS:
    print("Errors / skipped jobs (kept in raw/errors.json):")
    for entry in ERRORS:
        print("  ", entry)

## 12. Measurement results extraction
All results are rebuilt from the **raw counts** stored in `RUNS`. Each row below reports the measured $\hat P(K\ge3)$, its absolute and relative error with respect to the exact reference, the shot count, backend, job ID and physical qubits. Mitigated rows use readout-calibration circuits executed **in the same job** on the same qubits.

In [ ]:
def find_run(source, label, job_label=None):
    """Return the RunRecord for a source/label (and job label for hardware), or None."""
    if source == "hardware":
        return RUNS.get(f"hardware:{label}:{job_label}")
    return RUNS.get(f"{source}:{label}")


def calibration_runs_for(run):
    """Matching all-0 / all-1 records measured in the same job (or same simulation)."""
    suffix = run.label                                        # e.g. "A_best"
    if run.source == "hardware":
        zero = find_run("hardware", f"cal0_{suffix}", run.job_label)
        one = find_run("hardware", f"cal1_{suffix}", run.job_label)
    else:
        zero, one = find_run(run.source, f"cal0_{suffix}"), find_run(run.source, f"cal1_{suffix}")
    return (zero, one) if zero is not None and one is not None and run.suppression == "none" else (None, None)


def wilson_interval(successes, shots, confidence=0.95):
    interval = stats.binomtest(int(successes), int(shots)).proportion_ci(confidence_level=confidence, method="wilson")
    return interval.low, interval.high


def multinomial_resample(counts, generator):
    shots = sum(counts.values())
    return generator.multinomial(shots, counts_to_probability_vector(counts, NUM_CUSTOMERS)) / shots


def bootstrap_statistic(statistic, count_dicts, repetitions=1000, seed=RANDOM_SEED, confidence=0.95):
    """Percentile bootstrap of statistic(list_of_probability_vectors), resampling every counts dict."""
    generator = np.random.default_rng(seed)
    samples = np.array([statistic([multinomial_resample(c, generator) for c in count_dicts])
                        for _ in range(repetitions)])
    low, high = np.nanpercentile(samples, [100 * (1 - confidence) / 2, 100 * (1 + confidence) / 2])
    return float(low), float(high), samples


def raw_tail_row(name, run):
    shots = run.shots
    successes = round(tail_from_counts(run.counts) * shots)
    estimate = successes / shots
    low, high = wilson_interval(successes, shots)
    return {"name": name, "source": run.source, "layout": run.layout_name, "suppression": run.suppression,
            "mitigation": "none", "physical_qubits": run.physical_qubits, "backend": run.backend,
            "job_id": run.job_id, "shots": shots, "estimate": estimate, "ci_low": low, "ci_high": high}


def mitigated_tail_row(name, run, zero_run, one_run):
    vectors = lambda: [counts_to_probability_vector(c, NUM_CUSTOMERS) for c in (run.counts, zero_run.counts, one_run.counts)]
    point = tail_from_vector(mitigate_with_calibration(*vectors()))
    statistic = lambda v: tail_from_vector(mitigate_with_calibration(*v))
    low, high, _ = bootstrap_statistic(statistic, [run.counts, zero_run.counts, one_run.counts])
    return {"name": name, "source": run.source, "layout": run.layout_name, "suppression": run.suppression,
            "mitigation": "readout", "physical_qubits": run.physical_qubits, "backend": run.backend,
            "job_id": run.job_id, "shots": run.shots, "estimate": point, "ci_low": low, "ci_high": high}


def build_stage3_results_table():
    rows = []
    ideal = find_run("ideal_sim", "A")
    rows.append(raw_tail_row("Ideal Aer simulation", ideal))
    for layout in single_qubit_layouts:
        noisy = find_run("noisy_sim", f"A_{layout}")
        rows.append(raw_tail_row(f"Noisy Aer simulation ({layout} qubits)", noisy))
        zero, one = calibration_runs_for(noisy)
        if zero is not None:
            rows.append(mitigated_tail_row(f"Noisy Aer + readout mitigation ({layout})", noisy, zero, one))
    for key, run in RUNS.items():
        if run.source != "hardware" or run.experiment != "A":
            continue
        tag = {"none": "baseline", "dd_twirl": "DD + twirling"}[run.suppression]
        name = f"Hardware {tag} ({run.layout_name} qubits) [{run.job_label}]"
        rows.append(raw_tail_row(name, run))
        zero, one = calibration_runs_for(run)
        if zero is not None:
            rows.append(mitigated_tail_row(f"Hardware + readout mitigation ({run.layout_name}) [{run.job_label}]",
                                           run, zero, one))
    table = pd.DataFrame(rows)
    table["abs_error"] = (table.estimate - EXACT_TAIL).abs()
    table["signed_error"] = table.estimate - EXACT_TAIL
    table["rel_error_pct"] = 100 * table.abs_error / EXACT_TAIL
    standard_error = np.sqrt(EXACT_TAIL * (1 - EXACT_TAIL) / table.shots)
    table["z_vs_exact_shot_noise_only"] = table.signed_error / standard_error
    return table


stage3_table = build_stage3_results_table()
reference_rows = pd.DataFrame([
    {"name": "Exact: Poisson-binomial DP", "estimate": tail_dp},
    {"name": "Exact: enumeration of 64 outcomes", "estimate": tail_enumeration},
    {"name": "Validated reference (fixed)", "estimate": EXACT_REFERENCE_TAIL}])
print(reference_rows.to_string(index=False, float_format=lambda x: f"{x:.16f}"))
print()
display_columns = ["name", "shots", "estimate", "ci_low", "ci_high", "abs_error", "rel_error_pct", "job_id"]
print(stage3_table[display_columns].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nRaw hardware counts (Experiment A, first job):")
first_hardware = find_run("hardware", "A_best", "job1_stage3_best_layout")
if first_hardware is None:
    print("  no hardware result available in this run.")
else:
    print(f"  job {first_hardware.job_id} on {first_hardware.backend}, qubits {first_hardware.physical_qubits}, "
          f"{first_hardware.shots} shots")
    print("  ", dict(sorted(first_hardware.counts.items())))

## 13. Statistical analysis and uncertainty estimation
**Confidence intervals.** For $\hat P(K\ge 3)$ the shots are Bernoulli trials, so we use 95% Wilson intervals. For mitigated numbers and the Experiment B metrics, which are nonlinear functions of several count vectors, we use a 95% percentile bootstrap (multinomial resampling of every counts vector, seeded).

**What the intervals mean.** They capture *shot noise only*. Hardware has additional systematic error (readout, gate errors, drift) which is exactly what the comparison to the exact value measures. The column `z_vs_exact_shot_noise_only` shows how many shot-noise standard errors the measured value is from the exact value; values far from zero indicate systematic hardware error.

**Experiment B metrics** (for any distribution over bitstrings): *feasibility* = probability of a weight-3 outcome; *optimal-solution probability* = probability of the exact optimum; *approximation ratio* $=(C_{\text{worst feasible}}-\mathbb E_{\text{feas}}[C])/(C_{\text{worst feasible}}-C_{\text{opt}})$ (1 is perfect, about 0.5 for a random feasible choice); *regret* $=\mathbb E_{\text{feas}}[C]-C_{\text{opt}}$, the expected reward lost relative to the exact optimum among feasible shots.

In [ ]:
def selection_metric_statistic(metric_name, mitigate):
    """Statistic for the bootstrap: a Experiment-B metric from raw or readout-mitigated vectors."""
    def statistic(vectors):
        distribution = mitigate_with_calibration(*vectors) if mitigate else vectors[0]
        return selection_metrics(distribution)[metric_name]
    return statistic


B_METRICS = ["feasibility", "optimal_probability", "approximation_ratio", "regret"]


def evaluate_selection_run(name, run, mitigate=False):
    """Experiment-B metrics with bootstrap CIs for one RunRecord (raw or readout-mitigated)."""
    count_dicts = [run.counts]
    if mitigate:
        zero, one = calibration_runs_for(run)
        if zero is None:
            return None
        count_dicts += [zero.counts, one.counts]
    point_vectors = [counts_to_probability_vector(c, problem.size) for c in count_dicts]
    distribution = mitigate_with_calibration(*point_vectors) if mitigate else point_vectors[0]
    row = {"name": name, "source": run.source, "layout": run.layout_name, "suppression": run.suppression,
           "mitigation": "readout" if mitigate else "none", "job_id": run.job_id, "shots": run.shots,
           "physical_qubits": run.physical_qubits}
    row.update(selection_metrics(distribution))
    for metric in B_METRICS:
        low, high, _ = bootstrap_statistic(selection_metric_statistic(metric, mitigate), count_dicts, repetitions=500)
        row[f"{metric}_low"], row[f"{metric}_high"] = low, high
    return row


def build_selection_table():
    rows = []
    for name, metrics in baseline_metrics.items():
        rows.append({"name": f"Baseline: {name}", "source": "analytic", "layout": "-", "suppression": "none",
                     "mitigation": "none", "job_id": "", "shots": np.nan, **metrics})
    exact_qaoa = qaoa_ideal_results[QAOA_LAYERS]
    rows.append({"name": f"Ideal QAOA p={QAOA_LAYERS} (exact statevector)", "source": "analytic", "layout": "-",
                 "suppression": "none", "mitigation": "none", "job_id": "", "shots": np.nan,
                 **{m: exact_qaoa[m] for m in B_METRICS + ["optimal_probability_given_feasible", "mean_feasible_cost"]}})
    rows.append(evaluate_selection_run("Ideal Aer sampling", find_run("ideal_sim", "B")))
    for layout in chain_layouts:
        noisy = find_run("noisy_sim", f"B_{layout}")
        rows.append(evaluate_selection_run(f"Noisy Aer ({layout} chain)", noisy))
        rows.append(evaluate_selection_run(f"Noisy Aer + readout mitigation ({layout})", noisy, mitigate=True))
    for key, run in RUNS.items():
        if run.source == "hardware" and run.experiment == "B":
            tag = "baseline" if run.suppression == "none" else "DD + twirling"
            rows.append(evaluate_selection_run(f"Hardware {tag} ({run.label}; {run.layout_name} chain)", run))
            mitigated = evaluate_selection_run(f"Hardware + readout mitigation ({run.label}; {run.layout_name} chain)", run, mitigate=True)
            if mitigated is not None:
                rows.append(mitigated)
    return pd.DataFrame([row for row in rows if row is not None])


selection_table = build_selection_table()
print(selection_table[["name", "shots", "feasibility", "optimal_probability", "approximation_ratio", "regret"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

def build_qaoa_depth_comparison():
    """Same-instance p=1/p=2 comparison across exact, equal-shot simulation and hardware."""
    rows = []
    job_label = "job2_stage2_synthetic_best_chain"
    for layers, label in QAOA_DEPTH_LABELS.items():
        exact = {"depth": layers, "name": f"Ideal statevector QAOA p={layers}", "source": "ideal_statevector",
                 "layout": "best", "mitigation": "none", "job_id": "", "shots": 0}
        exact.update(selection_metrics(qaoa_ideal_results[layers]["probabilities"]))
        rows.append(exact)
        for source, name in (("ideal_sim", "Ideal Aer"), ("noisy_sim", "Backend-noisy Aer")):
            run = find_run(source, label)
            row = evaluate_selection_run(f"{name} p={layers}", run)
            row["depth"] = layers
            rows.append(row)
            if source == "noisy_sim":
                mitigated = evaluate_selection_run(f"{name} + readout mitigation p={layers}", run, mitigate=True)
                if mitigated is not None:
                    mitigated["depth"] = layers
                    rows.append(mitigated)
        hardware = find_run("hardware", label, job_label)
        if hardware is not None:
            row = evaluate_selection_run(f"IBM hardware p={layers}", hardware)
            row["depth"] = layers
            rows.append(row)
            mitigated = evaluate_selection_run(f"IBM hardware + readout mitigation p={layers}", hardware, mitigate=True)
            if mitigated is not None:
                mitigated["depth"] = layers
                rows.append(mitigated)
    return pd.DataFrame(rows)


qaoa_depth_comparison = build_qaoa_depth_comparison()
print("\nEqual-shot QAOA depth comparison (synthetic instance):")
print(qaoa_depth_comparison[["depth", "name", "shots", "feasibility", "optimal_probability", "approximation_ratio", "regret"]]
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ---- Hardware-specific statistics: Experiments C, D, E ------------------------------------------
hardware_stats = {}

# E: repeatability of Experiment A on the best qubits (baseline jobs only).
baseline_a_runs = [run for run in RUNS.values()
                   if run.source == "hardware" and run.label == "A_best" and run.suppression == "none"]
if len(baseline_a_runs) >= 2:
    contingency = np.array([[round(tail_from_counts(r.counts) * r.shots), r.shots - round(tail_from_counts(r.counts) * r.shots)]
                            for r in baseline_a_runs])
    chi2, p_value, _, _ = stats.chi2_contingency(contingency)
    estimates = [tail_from_counts(r.counts) for r in baseline_a_runs]
    hardware_stats["E_repeatability"] = {
        "jobs": [r.job_id for r in baseline_a_runs], "estimates": estimates,
        "spread_max_minus_min": max(estimates) - min(estimates), "chi2": float(chi2), "p_value": float(p_value),
        "interpretation": ("run-to-run differences are consistent with shot noise" if p_value > 0.05 else
                           "run-to-run differences exceed shot noise: calibration drift or other systematic variation")}
    print("E repeatability:", hardware_stats["E_repeatability"])
else:
    print("E repeatability: fewer than two baseline hardware runs available.")

# D (Stage 3): baseline vs suppression vs mitigation on the best qubits; paired bootstrap for mitigation.
baseline_job1 = find_run("hardware", "A_best", "job1_stage3_best_layout")
suppressed_a = find_run("hardware", "A_best", "job5_suppression")
if baseline_job1 is not None:
    zero, one = calibration_runs_for(baseline_job1)
    if zero is not None:
        def error_gain(vectors):          # positive = mitigation reduced |error|
            return abs(tail_from_vector(vectors[0]) - EXACT_TAIL) - abs(tail_from_vector(mitigate_with_calibration(*vectors)) - EXACT_TAIL)
        low, high, samples = bootstrap_statistic(error_gain, [baseline_job1.counts, zero.counts, one.counts])
        point = error_gain([counts_to_probability_vector(c, NUM_CUSTOMERS) for c in (baseline_job1.counts, zero.counts, one.counts)])
        hardware_stats["D_readout_mitigation_A"] = {
            "abs_error_reduction": point, "ci_low": low, "ci_high": high,
            "verdict": "improves" if low > 0 else ("worsens" if high < 0 else "inconclusive (CI includes 0)")}
        print("D readout mitigation on Experiment A:", hardware_stats["D_readout_mitigation_A"])
    if suppressed_a is not None:
        generator = np.random.default_rng(RANDOM_SEED)
        base_probability = counts_to_probability_vector(baseline_job1.counts, NUM_CUSTOMERS)
        supp_probability = counts_to_probability_vector(suppressed_a.counts, NUM_CUSTOMERS)
        differences = []
        for _ in range(1000):
            base_tail = tail_from_vector(generator.multinomial(baseline_job1.shots, base_probability) / baseline_job1.shots)
            supp_tail = tail_from_vector(generator.multinomial(suppressed_a.shots, supp_probability) / suppressed_a.shots)
            differences.append(abs(base_tail - EXACT_TAIL) - abs(supp_tail - EXACT_TAIL))
        low, high = np.percentile(differences, [2.5, 97.5])
        hardware_stats["D_dd_twirling_A"] = {
            "abs_error_reduction": float(abs(tail_from_counts(baseline_job1.counts) - EXACT_TAIL)
                                         - abs(tail_from_counts(suppressed_a.counts) - EXACT_TAIL)),
            "ci_low": float(low), "ci_high": float(high),
            "verdict": "improves" if low > 0 else ("worsens" if high < 0 else "inconclusive (CI includes 0)")}
        print("D DD+twirling on Experiment A:", hardware_stats["D_dd_twirling_A"])

# D / C (Experiment B): compare metric differences between selected rows.
def table_value(name_fragment, metric):
    match = selection_table[selection_table.name == name_fragment]
    return float(match.iloc[0][metric]) if len(match) else None


for layout in chain_layouts:
    base_name, supp_name = f"Hardware baseline ({layout} chain)", f"Hardware DD + twirling ({layout} chain)"
    base_ratio, supp_ratio = table_value(base_name, "approximation_ratio"), table_value(supp_name, "approximation_ratio")
    if base_ratio is not None and supp_ratio is not None:
        base_row = selection_table[selection_table.name == base_name].iloc[0]
        supp_row = selection_table[selection_table.name == supp_name].iloc[0]
        overlap = not (supp_row.approximation_ratio_low > base_row.approximation_ratio_high or
                       supp_row.approximation_ratio_high < base_row.approximation_ratio_low)
        hardware_stats[f"D_dd_twirling_B_{layout}"] = {
            "baseline_ratio": base_ratio, "suppressed_ratio": supp_ratio,
            "verdict": "inconclusive (intervals overlap)" if overlap else
                       ("improves" if supp_ratio > base_ratio else "worsens")}
        print(f"D DD+twirling on Experiment B ({layout}):", hardware_stats[f"D_dd_twirling_B_{layout}"])
if not hardware_stats:
    print("No hardware statistics available (simulation-only run).")

## 14. Research-quality graphs
Eight figures; each is saved as 300-dpi PNG and as PDF. Where hardware results are missing (simulation-only run), the figures show the simulation results and say so.

In [ ]:
def short_name(row_name):
    return (row_name.replace("Hardware", "HW").replace("simulation", "sim").replace("readout mitigation", "RO-mit")
            .replace("qubits", "q"))


def plot_tail_comparison(table):
    """Fig 1: exact vs ideal vs noisy vs hardware (with 95% CIs) and signed errors."""
    wanted = table[table.name.str.contains(r"Ideal Aer|Noisy Aer simulation \(best|Noisy Aer \+ readout mitigation \(best|Hardware baseline \(best.*job1|"
                                           r"Hardware \+ readout mitigation \(best.*job1|Hardware DD", regex=True)]
    if wanted.empty:
        wanted = table.head(6)
    colour_for = lambda n: (COLORS["ideal"] if "Ideal" in n else COLORS["noisy"] if "Noisy" in n else
                            COLORS["mitigated"] if "mitigation" in n else COLORS["suppressed"] if "DD" in n else COLORS["hardware"])
    fig, (ax_value, ax_error) = plt.subplots(1, 2, layout="constrained", figsize=(12, 4.6), sharey=True)
    positions = np.arange(len(wanted))
    for position, row in zip(positions, wanted.itertuples()):
        ax_value.errorbar(row.estimate, position, xerr=[[row.estimate - row.ci_low], [row.ci_high - row.estimate]],
                          fmt="o", color=colour_for(row.name), capsize=3, markersize=7)
        ax_error.errorbar(row.signed_error, position, xerr=[[row.estimate - row.ci_low], [row.ci_high - row.estimate]],
                          fmt="o", color=colour_for(row.name), capsize=3, markersize=7)
    ax_value.axvline(EXACT_TAIL, color=COLORS["exact"], linestyle="--", label=f"Exact = {EXACT_TAIL:.4f}")
    ax_error.axvline(0, color=COLORS["exact"], linestyle="--")
    ax_value.set_yticks(positions)
    ax_value.set_yticklabels([short_name(n) for n in wanted.name])
    ax_value.invert_yaxis()
    ax_value.set_ylim(len(wanted) - 0.5, -0.5)
    ax_value.set_xlabel(r"estimated $P(K\geq 3)$ (95% CI; axis does not start at 0)")
    ax_error.set_xlabel(r"signed error  $\hat P - P_{\mathrm{exact}}$ (95% CI)")
    ax_value.legend(loc="upper left")
    fig.suptitle("Stage 3: exact value vs ideal simulation vs noisy simulation vs IBM hardware")
    fig.supxlabel("Intervals show shot noise only; hardware systematic error is the distance from the dashed line.", fontsize=9)
    return fig


def plot_hardware_errors(table):
    """Fig 2: signed estimation error of every hardware / noisy-sim Stage 3 run."""
    subset = table[table.source.isin(["hardware", "noisy_sim"])]
    fig, ax = plt.subplots(layout="constrained", figsize=(9, 0.55 * max(len(subset), 4) + 1.5))
    if subset.empty:
        ax.text(0.5, 0.5, "No simulated or hardware runs available", ha="center"); ax.axis("off"); return fig
    for position, row in enumerate(subset.itertuples()):
        colour = COLORS["hardware"] if row.source == "hardware" else COLORS["noisy"]
        if row.mitigation == "readout":
            colour = COLORS["mitigated"]
        if row.suppression == "dd_twirl":
            colour = COLORS["suppressed"]
        ax.errorbar(row.signed_error, position, xerr=[[row.estimate - row.ci_low], [row.ci_high - row.estimate]],
                    fmt="o", color=colour, capsize=3)
    ax.axvline(0, color=COLORS["exact"], linestyle="--")
    shot_noise = 1.96 * math.sqrt(EXACT_TAIL * (1 - EXACT_TAIL) / SHOTS_STAGE3)
    ax.axvspan(-shot_noise, shot_noise, color=COLORS["neutral"], alpha=0.15, label=f"+/-1.96 sigma at {SHOTS_STAGE3} shots")
    ax.set_yticks(range(len(subset)))
    ax.set_yticklabels([short_name(n) for n in subset.name], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(r"signed error  $\hat P(K\geq3)-P_{\mathrm{exact}}$")
    ax.set_title("Stage 3 estimation error (95% CI, shot noise only)")
    ax.legend(loc="lower right")
    return fig


def plot_distributions():
    """Fig 3: distribution of K and of the most probable bitstrings."""
    sources = {"Exact": exact_outcomes}
    sources["Ideal sim"] = counts_to_probability_vector(find_run("ideal_sim", "A").counts, NUM_CUSTOMERS)
    sources["Noisy sim (best q)"] = counts_to_probability_vector(find_run("noisy_sim", "A_best").counts, NUM_CUSTOMERS)
    hardware_run = find_run("hardware", "A_best", "job1_stage3_best_layout")
    if hardware_run is not None:
        sources["Hardware (best q)"] = counts_to_probability_vector(hardware_run.counts, NUM_CUSTOMERS)
    palette = [COLORS["exact"], COLORS["ideal"], COLORS["noisy"], COLORS["hardware"]]
    fig, (ax_k, ax_bits) = plt.subplots(1, 2, layout="constrained", figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.8]})
    width = 0.8 / len(sources)
    for index, (name, vector) in enumerate(sources.items()):
        pmf = pmf_from_outcome_probabilities(vector)
        ax_k.bar(np.arange(NUM_CUSTOMERS + 1) + (index - len(sources) / 2 + 0.5) * width, pmf, width,
                 label=name, color=palette[index])
    ax_k.axvspan(SUCCESS_THRESHOLD - 0.5, NUM_CUSTOMERS + 0.5, color=COLORS["neutral"], alpha=0.12)
    ax_k.set_xlabel("number of subscribing customers K (shaded: K >= 3)")
    ax_k.set_ylabel("probability")
    ax_k.set_title("Distribution of K")
    ax_k.legend()
    top = np.argsort(-exact_outcomes)[:16]
    for index, (name, vector) in enumerate(sources.items()):
        ax_bits.bar(np.arange(len(top)) + (index - len(sources) / 2 + 0.5) * width, vector[top], width,
                    color=palette[index], label=name)
    ax_bits.set_xticks(range(len(top)))
    ax_bits.set_xticklabels([format(x, f"0{NUM_CUSTOMERS}b") for x in top], rotation=70, fontsize=8, family="monospace")
    ax_bits.set_xlabel("outcome bitstring  q5 q4 q3 q2 q1 q0  (16 most probable exact outcomes)")
    ax_bits.set_ylabel("probability")
    ax_bits.set_title("Measurement outcome distribution")
    fig.suptitle("Stage 3 measurement distributions")
    return fig


def plot_circuit_metrics():
    """Fig 4: depth and gate counts, logical vs transpiled."""
    labels = [p.label for p in PREPARED.values() if p.experiment in ("A", "B")]
    selected = [PREPARED[l] for l in labels]
    fig, axes = plt.subplots(1, 3, layout="constrained", figsize=(14, 4.2))
    x = np.arange(len(selected))
    axes[0].bar(x - 0.2, [p.metrics["logical_depth"] for p in selected], 0.4, label="logical", color=COLORS["neutral"])
    axes[0].bar(x + 0.2, [p.metrics["depth"] for p in selected], 0.4, label="transpiled", color=COLORS["hardware"])
    axes[0].set_title("Circuit depth")
    axes[1].bar(x - 0.2, [p.metrics["logical_two_qubit_gates"] for p in selected], 0.4, label="logical", color=COLORS["neutral"])
    axes[1].bar(x + 0.2, [p.metrics["two_qubit_gates"] for p in selected], 0.4, label="transpiled", color=COLORS["hardware"])
    axes[1].set_title(f"Two-qubit gates ({TWO_QUBIT_GATE.upper()} after transpilation)")
    axes[2].bar(x, [p.metrics["estimated_success_probability"] for p in selected], 0.6, color=COLORS["ideal"])
    axes[2].set_ylim(0, 1.0)
    axes[2].set_title("Estimated success probability\n(product of 1 - gate/readout error)")
    for axis in axes:
        axis.set_xticks(x)
        axis.set_xticklabels([p.label for p in selected], rotation=45, ha="right")
    axes[0].legend()
    fig.suptitle("Circuit resources: logical vs transpiled (no SWAP routing needed for any circuit)")
    return fig


def plot_calibration():
    """Fig 5: calibration of the compared physical qubit groups."""
    qubit_table = qubit_calibration.set_index("qubit")
    edge_error = {}
    for row in edge_calibration.itertuples():
        edge_error[(row.qubit_a, row.qubit_b)] = edge_error[(row.qubit_b, row.qubit_a)] = row.error
    fig, axes = plt.subplots(1, 3, layout="constrained", figsize=(14, 4.2))
    readout_errors = qubit_calibration.readout_error.dropna()
    axes[0].hist(readout_errors, bins=40, color=COLORS["neutral"], alpha=0.7)
    for (name, qubits), colour in zip(single_qubit_layouts.items(), (COLORS["mitigated"], COLORS["noisy"], COLORS["hardware"])):
        axes[0].axvline(qubit_table.loc[qubits, "readout_error"].mean(), color=colour, linewidth=2, label=f"{name} group mean")
    axes[0].set_xlabel("readout error"); axes[0].set_ylabel("number of qubits")
    axes[0].set_title("Readout error across the device"); axes[0].legend()
    names = list(single_qubit_layouts)
    axes[1].bar(np.arange(len(names)) - 0.2, [qubit_table.loc[single_qubit_layouts[n], "readout_error"].mean() for n in names],
                0.4, label="mean readout error", color=COLORS["hardware"])
    axes[1].bar(np.arange(len(names)) + 0.2, [qubit_table.loc[single_qubit_layouts[n], "sx_error"].mean() * 100 for n in names],
                0.4, label="mean sx error x 100", color=COLORS["ideal"])
    axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels([f"{n}\n{single_qubit_layouts[n]}" for n in names], fontsize=7)
    axes[1].set_title("Experiment A qubit groups"); axes[1].legend()
    chain_names = list(chain_layouts)
    for index, name in enumerate(chain_names):
        path = chain_layouts[name]
        errors = [edge_error[(a, b)] for a, b in zip(path[:-1], path[1:])]
        axes[2].bar(np.arange(len(errors)) + (index - 0.5) * 0.4, errors, 0.4, label=f"{name} chain {path}")
    axes[2].set_xticks(range(NUM_CUSTOMERS - 1)); axes[2].set_xticklabels([f"link {i}" for i in range(NUM_CUSTOMERS - 1)])
    axes[2].set_ylabel(f"{TWO_QUBIT_GATE.upper()} error"); axes[2].set_title("Experiment B chain links"); axes[2].legend(fontsize=7)
    fig.suptitle(f"Calibration data of {backend.name} for the compared qubit mappings"
                 f"{' (FAKE local snapshot)' if backend_is_fake else ''}")
    return fig


def plot_mitigation(table):
    """Fig 6: baseline vs mitigated vs suppressed (Stage 3 error; Experiment B approximation ratio)."""
    fig, (ax_a, ax_b) = plt.subplots(1, 2, layout="constrained", figsize=(13, 4.8))
    best_rows = table[table.layout == "best"]
    best_rows = best_rows[best_rows.source.isin(["hardware", "noisy_sim"])]
    if best_rows.empty:
        ax_a.text(0.5, 0.5, "no results", ha="center"); ax_a.axis("off")
    else:
        for position, row in enumerate(best_rows.itertuples()):
            colour = COLORS["mitigated"] if row.mitigation == "readout" else COLORS["suppressed"] if row.suppression == "dd_twirl" else \
                (COLORS["hardware"] if row.source == "hardware" else COLORS["noisy"])
            ax_a.errorbar(position, row.abs_error, yerr=[[row.abs_error - abs_interval(row)[0]], [abs_interval(row)[1] - row.abs_error]],
                          fmt="o", color=colour, capsize=3)
        ax_a.set_xlim(-0.6, len(best_rows) - 0.4); ax_a.set_xticks(range(len(best_rows))); ax_a.set_xticklabels([short_name(n) for n in best_rows.name], rotation=60, ha="right", fontsize=7)
        ax_a.set_ylabel(r"$|\hat P(K\geq3)-P_{\mathrm{exact}}|$")
        ax_a.set_title("Stage 3 on best qubits: baseline vs mitigation (lower is better)")
    subset = selection_table[selection_table.source.isin(["hardware", "noisy_sim"])]
    if subset.empty:
        ax_b.text(0.5, 0.5, "no results", ha="center"); ax_b.axis("off")
    else:
        for position, row in enumerate(subset.itertuples()):
            colour = COLORS["mitigated"] if row.mitigation == "readout" else COLORS["suppressed"] if row.suppression == "dd_twirl" else \
                (COLORS["hardware"] if row.source == "hardware" else COLORS["noisy"])
            ax_b.errorbar(position, row.approximation_ratio,
                          yerr=[[row.approximation_ratio - row.approximation_ratio_low], [row.approximation_ratio_high - row.approximation_ratio]],
                          fmt="o", color=colour, capsize=3)
        ax_b.axhline(baseline_metrics["random feasible"]["approximation_ratio"], color=COLORS["neutral"], linestyle=":", label="random feasible")
        ax_b.axhline(qaoa_ideal_results[QAOA_LAYERS]["approximation_ratio"], color=COLORS["exact"], linestyle="--", label="ideal QAOA")
        ax_b.set_xlim(-0.6, len(subset) - 0.4); ax_b.set_xticks(range(len(subset))); ax_b.set_xticklabels([short_name(n) for n in subset.name], rotation=60, ha="right", fontsize=7)
        ax_b.set_ylabel("approximation ratio (higher is better)")
        ax_b.set_title("Experiment B: baseline vs mitigation vs suppression")
        ax_b.legend()
    fig.suptitle("Does error suppression / mitigation help? (95% CI)")
    return fig


def abs_interval(row):
    """Interval of |error| obtained from the CI of the estimate."""
    low_error, high_error = row.ci_low - EXACT_TAIL, row.ci_high - EXACT_TAIL
    if low_error <= 0 <= high_error:
        return 0.0, max(abs(low_error), abs(high_error))
    return min(abs(low_error), abs(high_error)), max(abs(low_error), abs(high_error))


def plot_optimisation():
    """Fig 7: objective quality, feasibility and regret for Experiment B (SYNTHETIC instance)."""
    table = selection_table
    fig, axes = plt.subplots(1, 4, layout="constrained", figsize=(17, 5), sharey=True)
    metrics = [("feasibility", "feasibility (weight-3 probability)"), ("optimal_probability", "P(exact optimum)"),
               ("approximation_ratio", "approximation ratio"), ("regret", "regret (reward lost)")]
    colour_for = lambda n, s: (COLORS["neutral"] if s == "analytic" else COLORS["ideal"] if "Ideal" in n else
                               COLORS["noisy"] if "Noisy" in n else COLORS["hardware"])
    for axis, (metric, label) in zip(axes, metrics):
        for position, row in enumerate(table.itertuples()):
            value = getattr(row, metric)
            low, high = getattr(row, f"{metric}_low", np.nan), getattr(row, f"{metric}_high", np.nan)
            error = None if np.isnan(low) else [[value - low], [high - value]]
            axis.errorbar(value, position, xerr=error, fmt="o", color=colour_for(row.name, row.source), capsize=3)
        axis.set_xlabel(label)
    axes[0].set_yticks(range(len(table)))
    axes[0].set_yticklabels([short_name(n) for n in table.name], fontsize=8)
    axes[0].invert_yaxis()
    fig.suptitle("Experiment B (SYNTHETIC 6-candidate, choose-3 instance; not the Atomiq challenge problem)")
    return fig


def plot_qpu_usage():
    """Fig 8: QPU seconds per job (actual vs planned) and cumulative usage."""
    fig, ax = plt.subplots(layout="constrained", figsize=(9, 4.5))
    if not budget.ledger:
        ax.text(0.5, 0.5, "no hardware jobs were run", ha="center"); ax.axis("off"); return fig
    ledger = pd.DataFrame(budget.ledger)
    x = np.arange(len(ledger))
    ax.bar(x - 0.2, ledger.estimated_seconds, 0.4, label="planned (before run)", color=COLORS["neutral"])
    ax.bar(x + 0.2, ledger.counted_seconds, 0.4, label="actual QPU usage", color=COLORS["hardware"])
    ax.set_xticks(x); ax.set_xticklabels(ledger.job_label, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("QPU seconds per job")
    twin = ax.twinx()
    twin.plot(x, ledger.cumulative_seconds, color=COLORS["exact"], marker="o", label="cumulative")
    twin.axhline(budget.allowance, color=COLORS["exact"], linestyle="--", label=f"allowance {budget.allowance:.0f}s")
    twin.set_ylabel("cumulative QPU seconds"); twin.grid(False)
    handles = ax.get_legend_handles_labels()[0] + twin.get_legend_handles_labels()[0]
    labels = ax.get_legend_handles_labels()[1] + twin.get_legend_handles_labels()[1]
    ax.legend(handles, labels, loc="upper left")
    ax.set_title("QPU usage across jobs")
    return fig

In [ ]:
figures_to_make = [
    ("fig1_stage3_exact_vs_sim_vs_hardware", lambda: plot_tail_comparison(stage3_table)),
    ("fig2_stage3_estimation_errors", lambda: plot_hardware_errors(stage3_table)),
    ("fig3_stage3_distributions", plot_distributions),
    ("fig4_circuit_resources", plot_circuit_metrics),
    ("fig5_calibration_and_mappings", plot_calibration),
    ("fig6_mitigation_comparison", lambda: plot_mitigation(stage3_table)),
    ("fig7_optimisation_quality", plot_optimisation),
    ("fig8_qpu_usage", plot_qpu_usage),
]
for figure_name, make_figure in figures_to_make:
    try:
        figure = make_figure()
        save_figure(figure, figure_name)
        plt.show()
    except Exception as error:               # a plotting problem must never lose experimental data
        print(f"Figure {figure_name} could not be drawn: {type(error).__name__}: {error}")

## 15. Saving results and experimental artifacts
Raw measurements (`raw/`) are stored separately from processed results (`processed/`). The API key and the account object are never written. Failed or skipped jobs are preserved in `raw/errors.json`.

In [ ]:
def save_artifacts():
    # --- raw: counts exactly as measured/simulated, plus job ledger and errors ---
    for key, run in RUNS.items():
        file_name = key.replace(":", "__").replace("/", "_") + ".json"
        save_json(asdict(run), RAW_DIR / f"counts__{file_name}")
    save_json(job_ids(), RAW_DIR / "job_ledger.json")
    save_json(ERRORS, RAW_DIR / "errors.json")

    # --- processed ---
    save_json({"p_k_ge_3_dynamic_programming": tail_dp, "p_k_ge_3_enumeration": tail_enumeration,
               "validated_reference": EXACT_REFERENCE_TAIL, "pmf_dp": exact_pmf_dp, "pmf_enumeration": exact_pmf_enumeration,
               "outcome_probabilities": exact_outcomes, "stage3_probabilities": stage3_probabilities,
               "customer_ids": stage3_customer_ids, "probability_source": probability_source},
              PROCESSED_DIR / "exact_results.json")
    stage3_table.to_csv(PROCESSED_DIR / "stage3_results.csv", index=False)
    selection_table.to_csv(PROCESSED_DIR / "experiment_b_results.csv", index=False)
    qaoa_depth_comparison.to_csv(PROCESSED_DIR / "qaoa_hardware_depth_comparison.csv", index=False)
    metrics_table.to_csv(PROCESSED_DIR / "transpiled_circuit_metrics.csv", index=False)
    save_json({label: p.metrics for label, p in PREPARED.items()}, PROCESSED_DIR / "transpiled_circuit_metrics_full.json")
    pd.DataFrame(budget.ledger).to_csv(PROCESSED_DIR / "qpu_usage.csv", index=False)
    save_json(hardware_stats, PROCESSED_DIR / "hardware_statistics.json")
    save_json({"problem_is_synthetic": True, "values": problem.values, "neighbour_penalties": problem.neighbour_penalties,
               "select_count": problem.select_count, "optimal_outcome": optimal_outcome, "optimal_cost": optimal_cost,
               "qaoa_layers": QAOA_LAYERS, "gammas": qaoa_gammas, "betas": qaoa_betas,
               "depth_parameters": {str(k): {"gammas": v["gammas"], "betas": v["betas"]} for k, v in qaoa_ideal_results.items()},
               "hardware_depth_labels": QAOA_DEPTH_LABELS,
               "hardware_shots_per_depth": SHOTS_QAOA_HARDWARE,
               "hardware_calibration_shots_per_state": SHOTS_QAOA_HARDWARE_CALIBRATION,
               "ideal_metrics": {str(k): {m: v[m] for m in B_METRICS} for k, v in qaoa_ideal_results.items()}},
              PROCESSED_DIR / "experiment_b_synthetic_instance.json")
    count_rows = []
    for key, run in RUNS.items():
        for bitstring, count in sorted(run.counts.items()):
            count_rows.append({"record": key, "source": run.source, "experiment": run.experiment, "layout": run.layout_name,
                               "job_id": run.job_id, "shots": run.shots, "bitstring": bitstring.replace(" ", ""),
                               "count": count, "k_ones": bitstring.replace(" ", "").count("1")})
    pd.DataFrame(count_rows).to_csv(PROCESSED_DIR / "counts_long_format.csv", index=False)
    save_json(PROBABILITY_PROVENANCE, PROCESSED_DIR / "stage3_input_provenance.json")
    save_json(budget.summary(), PROCESSED_DIR / "qpu_budget_summary.json")
    save_json(budget.ledger, PROCESSED_DIR / "qpu_ledger.json")
    save_json(VALIDATION_LOG, PROCESSED_DIR / "validation_checks.json")

    # --- backend + calibration snapshot ---
    qubit_calibration.to_csv(PROCESSED_DIR / "backend_qubit_calibration.csv", index=False)
    edge_calibration.to_csv(PROCESSED_DIR / "backend_edge_calibration.csv", index=False)
    save_json({"name": backend.name, "num_qubits": backend.num_qubits, "is_fake_local_snapshot": backend_is_fake,
               "two_qubit_gate": TWO_QUBIT_GATE, "basis_gates": sorted(backend.target.operation_names),
               "layouts_experiment_a": single_qubit_layouts, "layouts_experiment_b": chain_layouts,
               "queue_pending_jobs_at_selection": None if backend_is_fake else backend.status().pending_jobs},
              PROCESSED_DIR / "backend_info.json")
    save_json({"started_utc": START_TIME_UTC, "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "python": platform.python_version(), "qiskit": qiskit.__version__, "qiskit_aer": qiskit_aer.__version__,
               "qiskit_ibm_runtime": qiskit_ibm_runtime.__version__, "random_seed": RANDOM_SEED,
               "shots": {"stage3": SHOTS_STAGE3, "calibration": SHOTS_CALIBRATION, "qaoa_simulation": SHOTS_QAOA, "qaoa_hardware_per_depth": SHOTS_QAOA_HARDWARE, "qaoa_hardware_calibration": SHOTS_QAOA_HARDWARE_CALIBRATION},
               "hardware_armed": HARDWARE_ENABLED, "new_jobs_submitted_this_run": NEW_JOBS_SUBMITTED,
               "qpu_seconds_counted": budget.used_seconds, "probability_source": probability_source},
              OUTPUT_DIR / "run_metadata.json")
    print(f"Artifacts saved under {OUTPUT_DIR.resolve()}")


save_artifacts()

# Safety net: make sure no artifact contains anything that looks like the API key.
secret_value = read_secret(API_KEY_SECRET_NAME)
if secret_value:
    leaked = [str(p) for p in OUTPUT_DIR.rglob("*") if p.is_file() and p.suffix in (".json", ".csv")
              and secret_value in p.read_text(errors="ignore")]
    print("API key found in saved artifacts:", leaked if leaked else "no (good)")
del secret_value

try:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("atomiq_ibm_hardware_results", "zip", OUTPUT_DIR)
    files.download(archive)
except ImportError:
    pass

## Final interpretation

In [ ]:
def interpret_results():
    lines = []
    lines.append(f"Exact P(K>=3) = {EXACT_TAIL:.6f} (DP and enumeration agree; matches the validated reference).")
    ideal = stage3_table[stage3_table.name == "Ideal Aer simulation"].iloc[0]
    lines.append(f"Ideal simulation: {ideal.estimate:.4f} [{ideal.ci_low:.4f}, {ideal.ci_high:.4f}] - consistent with exact, validating circuit and bit ordering.")
    hardware_rows = stage3_table[(stage3_table.source == "hardware") & (stage3_table.mitigation == "none") &
                                 (stage3_table.suppression == "none")]
    if hardware_rows.empty:
        lines.append("No hardware results in this run (no key / fake backend / disabled / jobs failed). Only simulations were executed; "
                     "do not report any hardware claim from this run.")
    else:
        for row in hardware_rows.itertuples():
            lines.append(f"Hardware {row.layout} qubits: {row.estimate:.4f} [{row.ci_low:.4f}, {row.ci_high:.4f}], "
                         f"|error| {row.abs_error:.4f} ({row.rel_error_pct:.1f}% relative), z = {row.z_vs_exact_shot_noise_only:+.1f} "
                         f"shot-noise sigmas, job {row.job_id}.")
        within = (hardware_rows.ci_low <= EXACT_TAIL) & (EXACT_TAIL <= hardware_rows.ci_high)
        lines.append(f"{int(within.sum())} of {len(hardware_rows)} hardware intervals contain the exact value; "
                     "any miss is systematic hardware error (mostly readout), not statistical fluctuation.")
    for key, value in hardware_stats.items():
        lines.append(f"{key}: {value.get('verdict', value.get('interpretation', value))}")
    b_hardware = selection_table[selection_table.source == "hardware"]
    if not b_hardware.empty:
        best_row = b_hardware.sort_values("approximation_ratio", ascending=False).iloc[0]
        lines.append(f"Experiment B (synthetic instance): best hardware approximation ratio {best_row.approximation_ratio:.3f} "
                     f"({best_row['name']}), feasibility {best_row.feasibility:.3f}, versus ideal "
                     f"{qaoa_ideal_results[QAOA_LAYERS]['approximation_ratio']:.3f} and random-feasible "
                     f"{baseline_metrics['random feasible']['approximation_ratio']:.3f}.")
    lines.append(f"QPU time counted: {budget.used_seconds:.1f} s of {budget.allowance:.0f} s allowance.")
    print("\n".join(f"- {line}" for line in lines))


interpret_results()

### How to read these results (and what not to claim)
* **Experiment A** is a genuine hardware validation of the Stage 3 estimate. The circuit has no entanglement, so the dominant hardware error is **readout error**, and readout mitigation is expected to help; check the Experiment D verdicts above rather than assuming it did.
* **Statistics:** the confidence intervals contain shot noise only. A hardware value far from the exact one is a systematic error and should be reported as such; the exact reference is never adjusted.
* **Experiment B uses synthetic data.** It demonstrates hardware-sized exact-cardinality optimisation (XY mixer, no SWAPs) against an exact classical optimum. It must **not** be described as solving the original ten-customer Stage 2 problem.
* **Experiments C, D, E** are single-day measurements on one device; intervals that overlap mean the comparison is **inconclusive**, not that a method is useless. Negative or inconclusive results are results.
* If a job was skipped by the budget guard, fewer comparisons are available; the notebook never extrapolates or fabricates missing hardware data.
* **Experiment A vs the data:** the six probabilities come from the project file recorded in `processed/stage3_input_provenance.json` (path, SHA-256, selection rule). The notebook refuses to run without a source that reproduces the validated reference.
* **Independence:** counts from different jobs on the same qubits are separate samples of the same drifting device, not independent draws of a fixed distribution. Readout-mitigated estimates reuse the calibration counts, so their bootstrap intervals include calibration noise but not drift between jobs.
* **Budget:** the planning balance comes from your dashboard value. Re-read the dashboard at the start of every session.
* **Mixer:** the sequential even/odd XY mixer is a product-formula approximation of global XY evolution (section 9b). It is a variational ansatz, and it conserves Hamming weight exactly.